In [1]:
"""IOD nonlinear CPU v3.1 -- complete CPU-only Google Colab experiments.

Copy this entire file into one Colab cell.  It deliberately uses CPU SciPy
HiGHS only: no Torch, CUDA, cuOpt, GPU, TPU, or encoded source payloads.

The implementation has two explicitly different utility modes:

``paper``
    A finite dictionary of literal positive powers ``x**m``.  This is the
    mode aligned with the supplied paper's assumption ``m_i > 0``.  For each
    simulated user, the generator first draws a local power band and then
    assigns every coordinate an independent random power from that band.

``signed_monotone``
    An extension for a signed range such as ``[-10, 10]``.  It uses a
    zero-safe, endpoint-normalized shifted Box--Cox basis.  It is monotone on
    benefit attributes but is *not* the paper's literal ``x**m`` family.

The IOD implementation is a finite-atom convex-hull LP relaxation. Comparison
experiments return the paper's formal safe outer set (near-optimal under at
least one feasible relaxed coefficient vector). It is not a certificate for a
continuous exponent interval with exactly one exponent per coordinate.
"""

from __future__ import annotations

# Set numerical-library thread limits before importing NumPy/SciPy. LP-
# relaxation tuple classifications are parallelized across processes; each HiGHS solve
# must remain single threaded to prevent oversubscription.
import os

for _thread_env in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ[_thread_env] = "1"
os.environ["HIGHS_THREADS"] = "1"

import hashlib
import json
import math
import multiprocessing as mp
import shutil
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass, replace
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable, Literal, Mapping, Sequence

import numpy as np
import pandas as pd
from scipy.optimize import OptimizeWarning, linprog



# ---------------------------------------------------------------------------
# EDIT ONLY THIS BLOCK IN COLAB
# ---------------------------------------------------------------------------
RUN_PROFILE: Literal["quick", "balanced", "paper"] = "paper"
QUICK = RUN_PROFILE == "quick"  # Backward-compatible internal flag.
UTILITY_MODE: Literal["paper", "signed_monotone"] = "paper"

# User-allowed positive literal powers. IOD retains this complete dictionary;
# the hidden utility does not use a fixed vector. For every case it selects a
# seeded random local band and independently draws each attribute's power from
# that band (controlled by TRUE_POWER_MAX_GAP below).
PAPER_POWER_CHOICES = tuple(float(p) for p in range(1, 11))

# The signed extension includes p=0, whose basis is the normalized log limit.
SIGNED_POWER_CHOICES = tuple(float(p) for p in range(-10, 11))
SIGNED_SHIFT = 1.0

# Hidden powers within one simulated utility may differ, but their largest
# pairwise difference is at most two. Examples include (2,3), (6,7), and
# (6,8). Under the primary protocol, the selected band is also the truthful
# answer to the paper's single exponent-range prompt; the coordinate-specific
# assignments remain hidden.
TRUE_POWER_MAX_GAP = 2.0

# Paper-faithful default: the simulated answer to the exponent-range prompt
# reveals the selected close band (for example [6,8]) but not the hidden
# coordinate-specific assignments.  ``full_dictionary_stress`` is retained as
# a separately labelled ablation; it is not the primary paper protocol.
EXPONENT_RANGE_POLICY: Literal[
    "elicited_band", "full_dictionary_stress"
] = "elicited_band"

WORKBOOK_PATH = Path("/content/Data/Real estate valuation data set.xlsx")
RESULT_ROOT = Path("/content/IOD_nonlinear_results")
N_WORKERS: int | None = None  # None -> min(4, CPU count), never fewer than 2 when possible.
RUN_SCALABILITY = True
SCALABILITY_SIZES = (1_000, 10_000, 100_000)
SCALABILITY_DIMENSION = 5
SCALABILITY_EPSILON = 0.05
# Full finite-LP inner/outer enumeration is deliberately limited to n=1,000.
# Larger n use exact-safe anytime bounds and must be labelled ``bounds_only``.
SCALABILITY_FULL_LP_MAX_N = 1_000
SCALABILITY_ANCHOR_DIRECTIONS = 32
SCALABILITY_LARGE_N_EXACT_CANDIDATES_PER_SET = 64
SCALABILITY_EXACT_REFINEMENT_SECONDS = 120.0

# The suite computes one canonical result bank and then exports separate views.
EXPERIMENT_STUDIES = (
    "iod_vs_squeeze",
    "all_original_algorithms",
    "budget_curves",
    "power_band_ablation",
    "scalability",
)
# ---------------------------------------------------------------------------


Array = np.ndarray
RELEASE_BUILD = "v3.1-20260723-concave-safe-r1"
METHOD_ORDER = (
    "Top-20 linear oracle weights",
    "Skyline",
    "Squeeze-u linear surrogate",
    "MinR sampled linear surrogate",
    "MinD sampled linear surrogate",
    "IOD (LP relaxation) safe outer",
)
CACHE_SCHEMA = "iod-nonlinear-cpu-v3-20260723-r3-safe-outer-only"
POWER_SAMPLING_POLICY = "seeded-local-band-conditional-iid-v1"
IOD_PREPRUNE_POLICY = "elicited-band-superhomogeneous-front-v1"
CONCAVE_CASE_REVISION = "concave-power-safe-front-v1"
LP_TOLERANCE = 1e-8


def _worker_count(requested: int | None = None) -> int:
    cpu = max(1, os.cpu_count() or 1)
    automatic = requested is None
    if automatic:
        requested = min(4, cpu)
    requested = max(1, min(4, int(requested), cpu))
    if automatic and cpu >= 2:
        requested = max(2, requested)
    return requested


@dataclass(frozen=True)
class ExperimentConfig:
    quick: bool
    profile: Literal["quick", "balanced", "paper"]
    utility_mode: Literal["paper", "signed_monotone"]
    powers: tuple[float, ...]
    truth_power_max_gap: float
    exponent_range_policy: Literal["elicited_band", "full_dictionary_stress"]
    signed_shift: float
    seeds: tuple[int, ...]
    synthetic_n: int
    dimensions: tuple[int, ...]
    epsilons: tuple[float, ...]
    adaptive_pairwise_questions: int
    baseline_pairwise_questions: int
    pair_pool_size: int
    matched_total_budgets: tuple[int, ...]
    matched_budget_dimension: int
    matched_budget_epsilon: float
    workers: int
    run_scalability: bool
    scalability_sizes: tuple[int, ...]
    scalability_dimension: int
    scalability_epsilon: float
    scalability_full_lp_max_n: int
    scalability_anchor_directions: int
    scalability_large_n_exact_candidates_per_set: int
    scalability_exact_refinement_seconds: float
    studies: tuple[str, ...]
    workbook_path: str
    result_root: str

    def validate(self) -> None:
        if self.utility_mode not in {"paper", "signed_monotone"}:
            raise ValueError("utility_mode must be paper or signed_monotone")
        if self.profile not in {"quick", "balanced", "paper"}:
            raise ValueError("profile must be quick, balanced, or paper")
        if self.exponent_range_policy not in {
            "elicited_band", "full_dictionary_stress"
        }:
            raise ValueError("invalid exponent_range_policy")
        p = np.asarray(self.powers, dtype=np.float64)
        if p.ndim != 1 or len(p) == 0 or not np.isfinite(p).all():
            raise ValueError("powers must be a nonempty finite vector")
        if len(np.unique(p)) != len(p):
            raise ValueError("powers must not contain duplicate atoms")
        if self.utility_mode == "paper" and np.any(p <= 0):
            raise ValueError("paper mode follows the PDF and requires every power > 0")
        if self.utility_mode == "signed_monotone" and (
            np.any(p < -10) or np.any(p > 10)
        ):
            raise ValueError("signed_monotone powers must stay in [-10,10]")
        if (
            not np.isfinite(self.truth_power_max_gap)
            or self.truth_power_max_gap < 0
        ):
            raise ValueError("truth_power_max_gap must be finite and nonnegative")
        if not np.isfinite(self.signed_shift) or self.signed_shift <= 0:
            raise ValueError("signed_shift must be finite and positive")
        if not self.seeds or len(self.seeds) != len(set(self.seeds)):
            raise ValueError("seeds must be nonempty and unique")
        if any(d < 2 for d in self.dimensions):
            raise ValueError("dimensions must be at least two")
        if any(e < 0 for e in self.epsilons):
            raise ValueError("epsilon must be nonnegative")
        if self.workers < 1 or self.workers > 4:
            raise ValueError("workers must lie in [1,4]")
        if any(int(n) < 1 for n in self.scalability_sizes):
            raise ValueError("scalability sizes must be positive")
        if self.scalability_dimension < 2:
            raise ValueError("scalability_dimension must be at least two")
        if self.scalability_epsilon < 0:
            raise ValueError("scalability_epsilon must be nonnegative")
        if self.scalability_full_lp_max_n < 1:
            raise ValueError("scalability_full_lp_max_n must be positive")
        if self.scalability_anchor_directions < 1:
            raise ValueError("scalability_anchor_directions must be positive")
        if self.scalability_large_n_exact_candidates_per_set < 0:
            raise ValueError(
                "scalability_large_n_exact_candidates_per_set must be nonnegative"
            )
        if self.scalability_exact_refinement_seconds < 0:
            raise ValueError("scalability_exact_refinement_seconds must be nonnegative")

    def hash_payload(self, *, workbook_sha256: str | None) -> dict[str, Any]:
        payload = asdict(self)
        payload["cache_schema"] = CACHE_SCHEMA
        payload["workbook_sha256"] = workbook_sha256
        return payload


def make_config() -> ExperimentConfig:
    powers = (
        tuple(map(float, PAPER_POWER_CHOICES))
        if UTILITY_MODE == "paper"
        else tuple(map(float, SIGNED_POWER_CHOICES))
    )
    if QUICK:
        config = ExperimentConfig(
            quick=True,
            profile="quick",
            utility_mode=UTILITY_MODE,
            powers=powers,
            truth_power_max_gap=float(TRUE_POWER_MAX_GAP),
            exponent_range_policy=EXPONENT_RANGE_POLICY,
            signed_shift=float(SIGNED_SHIFT),
            seeds=(20260722,),
            synthetic_n=120,
            dimensions=(5,),
            epsilons=(0.05,),
            adaptive_pairwise_questions=2,
            baseline_pairwise_questions=2,
            pair_pool_size=10,
            matched_total_budgets=(8,),
            matched_budget_dimension=5,
            matched_budget_epsilon=0.05,
            workers=_worker_count(N_WORKERS),
            run_scalability=False,
            scalability_sizes=(1_000,),
            scalability_dimension=5,
            scalability_epsilon=0.05,
            scalability_full_lp_max_n=1_000,
            scalability_anchor_directions=8,
            scalability_large_n_exact_candidates_per_set=0,
            scalability_exact_refinement_seconds=10.0,
            studies=tuple(EXPERIMENT_STUDIES),
            workbook_path=str(WORKBOOK_PATH),
            result_root=str(RESULT_ROOT),
        )
    else:
        seed_count = 5 if RUN_PROFILE == "balanced" else 10
        seeds = tuple(20260721 + offset for offset in range(seed_count))
        config = ExperimentConfig(
            quick=False,
            profile=RUN_PROFILE,
            utility_mode=UTILITY_MODE,
            powers=powers,
            truth_power_max_gap=float(TRUE_POWER_MAX_GAP),
            exponent_range_policy=EXPONENT_RANGE_POLICY,
            signed_shift=float(SIGNED_SHIFT),
            seeds=seeds,
            synthetic_n=1_000,
            dimensions=(5, 10, 15),
            epsilons=(0.10, 0.05, 0.01),
            adaptive_pairwise_questions=10,
            baseline_pairwise_questions=10,
            pair_pool_size=20,
            matched_total_budgets=(12, 15, 20, 25),
            matched_budget_dimension=5,
            matched_budget_epsilon=0.01,
            workers=_worker_count(N_WORKERS),
            run_scalability=bool(RUN_SCALABILITY),
            scalability_sizes=tuple(map(int, SCALABILITY_SIZES)),
            scalability_dimension=int(SCALABILITY_DIMENSION),
            scalability_epsilon=float(SCALABILITY_EPSILON),
            scalability_full_lp_max_n=int(SCALABILITY_FULL_LP_MAX_N),
            scalability_anchor_directions=int(SCALABILITY_ANCHOR_DIRECTIONS),
            scalability_large_n_exact_candidates_per_set=int(
                SCALABILITY_LARGE_N_EXACT_CANDIDATES_PER_SET
            ),
            scalability_exact_refinement_seconds=float(
                SCALABILITY_EXACT_REFINEMENT_SECONDS
            ),
            studies=tuple(EXPERIMENT_STUDIES),
            workbook_path=str(WORKBOOK_PATH),
            result_root=str(RESULT_ROOT),
        )
    config.validate()
    return config


def _jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, Mapping):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [_jsonable(item) for item in value]
    return value


def stable_hash(payload: Mapping[str, Any]) -> str:
    encoded = json.dumps(_jsonable(payload), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path: Path, payload: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(_jsonable(payload), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


# ---------------------------------------------------------------------------
# Utility features and heterogeneous ground truth
# ---------------------------------------------------------------------------


def validate_normalized(X: Any) -> Array:
    values = np.asarray(X, dtype=np.float64)
    if values.ndim != 2 or values.shape[0] == 0 or values.shape[1] == 0:
        raise ValueError("X must be a nonempty n-by-d matrix")
    if not np.isfinite(values).all():
        raise ValueError("X contains nonfinite values")
    if np.any(values < -1e-12) or np.any(values > 1 + 1e-12):
        raise ValueError("X must be normalized to [0,1]")
    return np.clip(values, 0.0, 1.0)


def signed_monotone_basis(
    X: Any, powers: Sequence[float], *, shift: float = 1.0
) -> Array:
    """Endpoint-normalized shifted Box--Cox atoms, including the p=0 log limit.

    For p != 0 this equals

        expm1(p*log(1+x/s)) / expm1(p*log(1+1/s)).

    Stable positive/negative rearrangements keep the result finite at x=0 and
    in [0,1].  Every atom is nondecreasing for p in [-10,10].
    """

    values = np.asarray(X, dtype=np.float64)
    p = np.asarray(powers, dtype=np.float64)
    if p.ndim != 1 or len(p) == 0 or not np.isfinite(p).all():
        raise ValueError("powers must be a nonempty finite vector")
    if np.any(p < -10) or np.any(p > 10):
        raise ValueError("signed powers must lie in [-10,10]")
    if not np.isfinite(shift) or shift <= 0:
        raise ValueError("shift must be finite and positive")
    if not np.isfinite(values).all() or np.any(values < -1e-12) or np.any(values > 1+1e-12):
        raise ValueError("signed basis inputs must lie in [0,1]")
    values = np.clip(values, 0.0, 1.0)
    log_x = np.log1p(values[..., None] / float(shift))
    log_one = float(np.log1p(1.0 / float(shift)))
    output_shape = values.shape + (len(p),)
    log_x = np.broadcast_to(log_x, output_shape)
    pp = np.broadcast_to(p, output_shape)
    out = np.empty(output_shape, dtype=np.float64)
    near_zero = np.abs(pp * log_one) <= np.sqrt(np.finfo(np.float64).eps)
    out[near_zero] = (log_x / log_one)[near_zero]
    positive = (~near_zero) & (pp > 0)
    if np.any(positive):
        z = pp[positive] * log_x[positive]
        Z = pp[positive] * log_one
        out[positive] = np.exp(z-Z) * (-np.expm1(-z)) / (-np.expm1(-Z))
    negative = (~near_zero) & (pp < 0)
    if np.any(negative):
        z = pp[negative] * log_x[negative]
        Z = pp[negative] * log_one
        out[negative] = (-np.expm1(z)) / (-np.expm1(Z))
    out = np.clip(out, 0.0, 1.0)
    out[np.broadcast_to(values[..., None] == 0.0, out.shape)] = 0.0
    out[np.broadcast_to(values[..., None] == 1.0, out.shape)] = 1.0
    return out


def feature_matrix(
    X: Any,
    powers: Sequence[float],
    *,
    mode: Literal["paper", "signed_monotone"],
    signed_shift: float,
) -> Array:
    values = validate_normalized(X)
    p = np.asarray(powers, dtype=np.float64)
    if mode == "paper":
        if p.ndim != 1 or len(p) == 0 or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError("paper powers must be finite and strictly positive")
        features = np.power(values[:, :, None], p[None, None, :])
    elif mode == "signed_monotone":
        features = signed_monotone_basis(values, p, shift=signed_shift)
    else:
        raise ValueError(mode)
    return np.ascontiguousarray(features.reshape(len(values), -1), dtype=np.float64)


@dataclass(frozen=True)
class TrueUtility:
    weights: Array
    powers: Array
    theta: Array
    power_indices: Array
    selected_power_band: Array | None = None


def make_true_utility(
    d: int,
    powers: Sequence[float],
    *,
    seed: int,
    max_power_gap: float = 2.0,
    forced_band: Sequence[float] | None = None,
) -> TrueUtility:
    dictionary = np.asarray(powers, dtype=np.float64)
    if d < 1:
        raise ValueError("d must be positive")
    if (
        dictionary.ndim != 1
        or len(dictionary) == 0
        or not np.isfinite(dictionary).all()
    ):
        raise ValueError("powers must be a nonempty finite vector")
    if len(np.unique(dictionary)) != len(dictionary):
        raise ValueError("powers must not contain duplicate atoms")
    if not np.isfinite(max_power_gap) or max_power_gap < 0:
        raise ValueError("max_power_gap must be finite and nonnegative")
    seed_sequence = np.random.SeedSequence(int(seed))
    weight_seed, band_seed, assignment_seed = seed_sequence.spawn(3)
    weight_rng = np.random.default_rng(weight_seed)
    band_rng = np.random.default_rng(band_seed)
    assignment_rng = np.random.default_rng(assignment_seed)
    weights = weight_rng.dirichlet(np.ones(d)).astype(np.float64)

    # Build all forward local bands whose numeric span respects max_power_gap.
    # Isolated dictionary values remain valid singleton bands, so every allowed
    # power can still be selected. The band itself and every coordinate draw
    # are seeded; rerunning a case reproduces exactly the same hidden utility.
    ordered = np.argsort(dictionary, kind="stable")
    bands: list[Array] = []
    covered_by_multi = np.zeros(len(dictionary), dtype=bool)
    tolerance = 1e-12
    for start in range(len(ordered)):
        first = dictionary[ordered[start]]
        stop = start
        while (
            stop + 1 < len(ordered)
            and dictionary[ordered[stop + 1]] - first
            <= float(max_power_gap) + tolerance
        ):
            stop += 1
        band = ordered[start : stop + 1]
        if len(band) >= 2:
            bands.append(band.copy())
            covered_by_multi[band] = True
    for index in ordered:
        if not covered_by_multi[int(index)]:
            bands.append(np.asarray([int(index)], dtype=int))
    if not bands:
        raise AssertionError("power-band construction produced no candidates")

    if forced_band is None:
        band = bands[int(band_rng.integers(0, len(bands)))]
    else:
        requested = np.asarray(forced_band, dtype=np.float64)
        if requested.ndim != 1 or len(requested) == 0:
            raise ValueError("forced_band must be a nonempty vector")
        if float(np.ptp(requested)) > float(max_power_gap)+tolerance:
            raise ValueError("forced_band exceeds max_power_gap")
        indices_in_dictionary = [
            int(np.flatnonzero(np.isclose(dictionary, value, atol=0.0, rtol=0.0))[0])
            for value in requested
            if np.any(np.isclose(dictionary, value, atol=0.0, rtol=0.0))
        ]
        if len(indices_in_dictionary) != len(requested):
            raise ValueError("forced_band contains an atom outside the dictionary")
        band = np.asarray(indices_in_dictionary, dtype=int)
    indices = assignment_rng.choice(band, size=int(d), replace=True).astype(int)
    chosen = dictionary[indices]
    if float(np.ptp(chosen)) > float(max_power_gap) + tolerance:
        raise AssertionError("sampled true powers exceed max_power_gap")
    theta = np.zeros(d * len(dictionary), dtype=np.float64)
    for attribute, (weight, atom) in enumerate(zip(weights, indices)):
        theta[attribute * len(dictionary) + int(atom)] = float(weight)
    return TrueUtility(
        weights,
        chosen,
        theta,
        indices.astype(int),
        dictionary[band].copy(),
    )


def utility_scores(
    X: Any,
    truth: TrueUtility,
    dictionary: Sequence[float],
    *,
    mode: Literal["paper", "signed_monotone"],
    signed_shift: float,
) -> Array:
    Phi = feature_matrix(X, dictionary, mode=mode, signed_shift=signed_shift)
    return Phi @ truth.theta


def target_indices(scores: Sequence[float], epsilon: float) -> Array:
    values = np.asarray(scores, dtype=np.float64)
    return np.flatnonzero((1.0 + float(epsilon)) * values >= values.max() - 1e-12)


def generate_anti_correlated(n: int, d: int, *, seed: int) -> Array:
    rng = np.random.default_rng(int(seed))
    base = rng.random((int(n), 1))
    if d == 1:
        return base
    noise = 0.05 * rng.standard_normal((int(n), int(d) - 1))
    return np.clip(np.concatenate([base, 1.0-base+noise], axis=1), 0.0, 1.0)


def _find_workbook(requested: Path) -> Path:
    if requested.exists():
        return requested
    parent = requested.parent
    if parent.exists():
        candidates = sorted(
            (p for p in parent.iterdir() if p.is_file() and p.suffix.casefold() in {".xlsx", ".xls"}),
            key=lambda p: p.name.casefold(),
        )
        exact = [p for p in candidates if p.stem.casefold() == "real estate valuation data set"]
        if exact:
            return exact[0]
    raise FileNotFoundError(
        f"Expected the 414-row workbook at {requested}. Upload it under /content/Data."
    )


def load_real_estate(path: str | Path) -> tuple[Array, pd.DataFrame, pd.DataFrame]:
    workbook = _find_workbook(Path(path))
    raw = pd.read_excel(workbook)
    # Normalize headers because the UCI workbook sometimes has leading spaces.
    raw.columns = [str(column).strip() for column in raw.columns]
    aliases = {
        "age": next((c for c in raw if "house age" in c.casefold()), None),
        "distance": next((c for c in raw if "distance" in c.casefold() and "mrt" in c.casefold()), None),
        "stores": next((c for c in raw if "convenience stores" in c.casefold()), None),
        "price": next((c for c in raw if "house price" in c.casefold()), None),
    }
    if any(value is None for value in aliases.values()):
        raise ValueError(f"Workbook columns do not match the UCI schema: {list(raw.columns)}")
    selected = pd.DataFrame({key: pd.to_numeric(raw[column], errors="coerce") for key, column in aliases.items()})
    if len(selected) != 414:
        raise ValueError(f"Expected 414 records, found {len(selected)}")
    if selected.isna().any().any():
        selected = selected.fillna(selected.median(numeric_only=True))

    def benefit(series: pd.Series) -> Array:
        lo, hi = float(series.min()), float(series.max())
        return np.zeros(len(series)) if hi == lo else (series.to_numpy()-lo)/(hi-lo)

    def cost(series: pd.Series) -> Array:
        return 1.0-benefit(series)

    processed = pd.DataFrame(
        {
            "newer": cost(selected["age"]),
            "nearer_mrt": cost(selected["distance"]),
            "stores": benefit(selected["stores"]),
            "lower_price": cost(selected["price"]),
        }
    )
    processed.insert(0, "row_id", np.arange(1, len(processed)+1))
    X = processed[["newer", "nearer_mrt", "stores", "lower_price"]].to_numpy(dtype=np.float64)
    if X.shape != (414, 4):
        raise AssertionError(X.shape)
    return validate_normalized(X), raw, processed


# ---------------------------------------------------------------------------
# Finite-atom convex-hull coefficient polytope and IOD LP relaxation
# ---------------------------------------------------------------------------


def _linprog(
    objective: Array,
    A_ub: Array | None,
    b_ub: Array | None,
    *,
    maximize: bool = False,
) -> tuple[bool, float | None, Array | None, int, str]:
    c = -np.asarray(objective, dtype=np.float64) if maximize else np.asarray(objective, dtype=np.float64)
    K = len(c)
    original_A = None if A_ub is None else np.asarray(A_ub, dtype=np.float64)
    original_b = None if b_ub is None else np.asarray(b_ub, dtype=np.float64)

    def run_once(
        method: Literal["highs", "highs-ds", "highs-ipm"],
        *,
        presolve: bool,
        scaled: bool,
        tolerance: float,
    ) -> tuple[bool, float | None, Array | None, int, str]:
        solve_A, solve_b = original_A, original_b
        if scaled and original_A is not None and original_b is not None and len(original_A):
            row_scale = np.maximum(
                np.max(np.abs(original_A), axis=1), np.abs(original_b)
            )
            row_scale = np.where(row_scale > 1e-12, row_scale, 1.0)
            solve_A = original_A / row_scale[:, None]
            solve_b = original_b / row_scale
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r"Unrecognized options detected: .*threads.*",
                category=OptimizeWarning,
            )
            result = linprog(
                c,
                A_ub=solve_A,
                b_ub=solve_b,
                A_eq=np.ones((1, K), dtype=np.float64),
                b_eq=np.ones(1, dtype=np.float64),
                bounds=[(0.0, None)] * K,
                method=method,
                options={
                    "presolve": bool(presolve),
                    "threads": 1,
                    "primal_feasibility_tolerance": float(tolerance),
                    "dual_feasibility_tolerance": float(tolerance),
                    "ipm_optimality_tolerance": max(1e-10, float(tolerance)),
                },
            )
        if not result.success or result.x is None or result.fun is None:
            return False, None, None, int(result.status), str(result.message)
        value = -float(result.fun) if maximize else float(result.fun)
        x = np.asarray(result.x, dtype=np.float64)
        # Validate every successful retry against the original, unscaled LP.
        violation = abs(float(x.sum()) - 1.0)
        violation = max(violation, max(0.0, -float(x.min())))
        if original_A is not None and original_b is not None and len(original_A):
            violation = max(violation, float(np.max(original_A @ x - original_b)))
        if violation > 5e-7:
            return False, None, None, 4, f"primal residual {violation:g}"
        return True, value, x, int(result.status), str(result.message)

    first = run_once("highs", presolve=True, scaled=False, tolerance=1e-9)
    if first[0] or first[3] != 4:
        return first

    # HiGHS occasionally reports model_status=Unknown/primal_status=Infeasible
    # for a numerically borderline LP. Retry only that ambiguous status, using
    # row scaling and two distinct algorithms. Normal optimal/infeasible calls
    # therefore pay no retry overhead.
    attempts = [first]
    for method in ("highs-ds", "highs-ipm"):
        retry = run_once(method, presolve=False, scaled=True, tolerance=1e-8)
        attempts.append(retry)
        if retry[0]:
            return retry
    retry_statuses = [attempt[3] for attempt in attempts[1:]]
    if retry_statuses and all(status == 2 for status in retry_statuses):
        return False, None, None, 2, "both scaled HiGHS retries found the LP infeasible"
    details = " | ".join(
        f"status {status}: {message}" for _, _, _, status, message in attempts
    )
    return False, None, None, 4, f"ambiguous after HiGHS retries: {details}"


def _phase_one_feasibility(
    A_ub: Array | None,
    b_ub: Array | None,
    K: int,
) -> tuple[Literal["feasible", "infeasible", "ambiguous"], Array | None, str]:
    """Resolve an ambiguous feasibility LP using a nonnegative violation slack.

    The phase-I model is always feasible and minimizes the largest inequality
    violation. A successful zero-slack solution supplies a checked witness;
    positive optimal slack certifies infeasibility at the declared tolerance.
    """

    if A_ub is None or b_ub is None or len(A_ub) == 0:
        return "feasible", np.full(K, 1.0 / K, dtype=np.float64), "no inequalities"
    A = np.asarray(A_ub, dtype=np.float64)
    b = np.asarray(b_ub, dtype=np.float64)
    augmented_A = np.column_stack([A, -np.ones(len(A), dtype=np.float64)])
    objective = np.zeros(K + 1, dtype=np.float64)
    objective[-1] = 1.0
    messages: list[str] = []
    for method in ("highs-ds", "highs-ipm"):
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r"Unrecognized options detected: .*threads.*",
                category=OptimizeWarning,
            )
            result = linprog(
                objective,
                A_ub=augmented_A,
                b_ub=b,
                A_eq=np.concatenate([np.ones(K), np.zeros(1)])[None, :],
                b_eq=np.ones(1, dtype=np.float64),
                bounds=[(0.0, None)] * (K + 1),
                method=method,
                options={
                    "presolve": False,
                    "threads": 1,
                    "primal_feasibility_tolerance": 1e-8,
                    "dual_feasibility_tolerance": 1e-8,
                    "ipm_optimality_tolerance": 1e-9,
                },
            )
        if not result.success or result.x is None or result.fun is None:
            messages.append(f"{method} status {result.status}: {result.message}")
            continue
        x = np.asarray(result.x[:-1], dtype=np.float64)
        slack = max(0.0, float(result.x[-1]), float(result.fun))
        residual = max(
            abs(float(x.sum()) - 1.0),
            max(0.0, -float(x.min())),
            float(np.max(A @ x - b)) - slack,
        )
        if residual > 5e-7:
            messages.append(f"{method} phase-I residual {residual:g}")
            continue
        if slack <= 5e-7:
            return "feasible", x, f"{method} phase-I slack {slack:g}"
        return "infeasible", None, f"{method} phase-I optimum slack {slack:g}"
    return "ambiguous", None, " | ".join(messages)


@dataclass
class CoefficientPolytope:
    d: int
    L: int
    A_ub: list[Array]
    b_ub: list[float]

    @property
    def K(self) -> int:
        return int(self.d * self.L)

    def arrays(self) -> tuple[Array | None, Array | None]:
        if not self.A_ub:
            return None, None
        return np.vstack(self.A_ub), np.asarray(self.b_ub, dtype=np.float64)

    def clone(self) -> "CoefficientPolytope":
        return CoefficientPolytope(
            self.d,
            self.L,
            [np.asarray(row, dtype=np.float64).copy() for row in self.A_ub],
            list(map(float, self.b_ub)),
        )

    def add_le(self, row: Sequence[float], bound: float) -> None:
        values = np.asarray(row, dtype=np.float64)
        if values.shape != (self.K,):
            raise ValueError("polytope constraint has the wrong width")
        self.A_ub.append(values.copy())
        self.b_ub.append(float(bound))

    def solve(self, objective: Sequence[float], *, maximize: bool = False) -> tuple[float, Array]:
        A, b = self.arrays()
        success, value, x, _, message = _linprog(
            np.asarray(objective, dtype=np.float64), A, b, maximize=maximize
        )
        if not success or value is None or x is None:
            raise RuntimeError(f"LP failed: {message}")
        return value, x

    def feasible_point(self) -> Array:
        return self.solve(np.zeros(self.K))[1]

    def contains(self, theta: Sequence[float], tolerance: float = 1e-7) -> bool:
        values = np.asarray(theta, dtype=np.float64)
        if values.shape != (self.K,) or values.min() < -tolerance:
            return False
        if abs(float(values.sum()) - 1.0) > tolerance:
            return False
        A, b = self.arrays()
        return A is None or float(np.max(A @ values - b)) <= tolerance


def _mass_vector(attribute: int, d: int, L: int) -> Array:
    row = np.zeros(d * L, dtype=np.float64)
    row[attribute * L : (attribute + 1) * L] = 1.0
    return row


def _ratio_bin(value: float, edges: Sequence[float]) -> tuple[float, float]:
    bins = np.asarray(edges, dtype=np.float64)
    position = int(np.searchsorted(bins, value, side="right") - 1)
    position = max(0, min(position, len(bins)-2))
    return float(bins[position]), float(bins[position+1])


def initial_polytope(
    truth: TrueUtility,
    d: int,
    L: int,
    *,
    dictionary: Sequence[float] | None = None,
    exponent_range_policy: Literal[
        "elicited_band", "full_dictionary_stress"
    ] = "elicited_band",
    ratio_edges: Sequence[float] = (0.0, 0.25, 0.50, 0.75, 1.0),
) -> tuple[CoefficientPolytope, tuple[int, ...], int]:
    """Simulate the paper-inspired d+2 initialization prompts.

    The primary protocol treats the selected close band as the truthful global
    exponent-range answer and removes atoms outside that band.  It never
    reveals which allowed atom a particular coordinate uses.  The separately
    labelled ``full_dictionary_stress`` policy deliberately keeps every atom.
    One baseline and d attribute swing answers reveal a coarse weight-mass
    order and ratio bins.  They do not reveal the hidden per-coordinate powers.
    Prompt counts are reported, but different prompt formats need not require
    equal respondent effort.
    """

    polytope = CoefficientPolytope(d, L, [], [])
    if exponent_range_policy not in {
        "elicited_band", "full_dictionary_stress"
    }:
        raise ValueError("invalid exponent_range_policy")
    if exponent_range_policy == "elicited_band":
        atoms = np.asarray(dictionary, dtype=np.float64)
        if atoms.shape != (L,):
            raise ValueError("dictionary does not match the atom dimension")
        if truth.selected_power_band is None or not len(truth.selected_power_band):
            raise ValueError("elicited_band requires a nonempty selected band")
        allowed = np.isin(atoms, np.asarray(truth.selected_power_band, dtype=np.float64))
        if not np.any(allowed):
            raise ValueError("the elicited exponent band contains no configured atom")
        for attribute in range(d):
            for atom_index in np.flatnonzero(~allowed):
                row = np.zeros(d * L, dtype=np.float64)
                row[attribute * L + int(atom_index)] = 1.0
                polytope.add_le(row, 0.0)
    masses = np.asarray(truth.weights, dtype=np.float64)
    order = tuple(np.argsort(-masses, kind="stable").tolist())
    reference = int(order[0])
    for preferred, other in zip(order[:-1], order[1:]):
        polytope.add_le(
            -_mass_vector(preferred, d, L) + _mass_vector(other, d, L), 0.0
        )
    reference_vector = _mass_vector(reference, d, L)
    for attribute in range(d):
        if attribute == reference:
            continue
        ratio = float(masses[attribute] / masses[reference])
        lower, upper = _ratio_bin(ratio, ratio_edges)
        attribute_vector = _mass_vector(attribute, d, L)
        polytope.add_le(attribute_vector-upper*reference_vector, 0.0)
        polytope.add_le(-attribute_vector+lower*reference_vector, 0.0)
    if not polytope.contains(truth.theta):
        raise AssertionError("simulated initialization removed the true utility")
    polytope.feasible_point()
    return polytope, order, d+2


@dataclass(frozen=True)
class PairQuestion:
    a: int
    b: int
    lower: float
    upper: float
    width: float


def _anchor_pool(
    Phi: Array,
    polytope: CoefficientPolytope,
    *,
    pool_size: int,
    seed: int,
) -> Array:
    """Small deterministic real-tuple pool from feasible utility anchors."""

    anchors = [polytope.feasible_point()]
    rng = np.random.default_rng(int(seed))
    for _ in range(6):
        direction = rng.standard_normal(polytope.K)
        try:
            anchors.append(polytope.solve(direction, maximize=True)[1])
        except RuntimeError:
            continue
    score_matrix = np.vstack(anchors) @ Phi.T
    selected: list[int] = []
    for scores in score_matrix:
        for index in np.argsort(-scores, kind="stable")[:3]:
            if int(index) not in selected:
                selected.append(int(index))
            if len(selected) >= pool_size:
                return np.asarray(selected, dtype=int)
    variability = np.ptp(score_matrix, axis=0)
    for index in np.argsort(-variability, kind="stable"):
        if int(index) not in selected:
            selected.append(int(index))
        if len(selected) >= min(pool_size, len(Phi)):
            break
    return np.asarray(selected, dtype=int)


def select_pair_question(
    Phi: Array,
    polytope: CoefficientPolytope,
    *,
    pool_size: int,
    asked: set[tuple[int, int]],
    seed: int,
) -> PairQuestion | None:
    pool = _anchor_pool(Phi, polytope, pool_size=pool_size, seed=seed)
    best_crossing: PairQuestion | None = None
    best_any: PairQuestion | None = None
    for a, b in combinations(pool.tolist(), 2):
        key = tuple(sorted((int(a), int(b))))
        if key in asked or np.array_equal(Phi[a], Phi[b]):
            continue
        delta = Phi[a]-Phi[b]
        lower = polytope.solve(delta, maximize=False)[0]
        upper = polytope.solve(delta, maximize=True)[0]
        question = PairQuestion(int(a), int(b), lower, upper, max(0.0, upper-lower))
        if question.width <= LP_TOLERANCE:
            continue
        if best_any is None or (question.width, -question.a, -question.b) > (
            best_any.width, -best_any.a, -best_any.b
        ):
            best_any = question
        if lower < -LP_TOLERANCE and upper > LP_TOLERANCE:
            if best_crossing is None or (question.width, -question.a, -question.b) > (
                best_crossing.width, -best_crossing.a, -best_crossing.b
            ):
                best_crossing = question
    return best_crossing if best_crossing is not None else best_any


@dataclass
class IODInteraction:
    Phi: Array
    polytope: CoefficientPolytope
    mandatory_questions: int
    pairwise_questions: int
    trace: list[dict[str, Any]]
    interaction_seconds: float


def run_iod_interaction(
    X: Array,
    truth: TrueUtility,
    config: ExperimentConfig,
    *,
    adaptive_questions: int,
    seed: int,
) -> IODInteraction:
    # Report the complete IOD relaxation cost, including feature construction
    # and the simulated initialization constraints.
    started = time.perf_counter()
    Phi = feature_matrix(
        X,
        config.powers,
        mode=config.utility_mode,
        signed_shift=config.signed_shift,
    )
    polytope, _, mandatory = initial_polytope(
        truth,
        X.shape[1],
        len(config.powers),
        dictionary=config.powers,
        exponent_range_policy=config.exponent_range_policy,
    )
    asked: set[tuple[int, int]] = set()
    trace: list[dict[str, Any]] = []
    for round_number in range(1, int(adaptive_questions)+1):
        question = select_pair_question(
            Phi,
            polytope,
            pool_size=config.pair_pool_size,
            asked=asked,
            seed=int(seed)+104729*round_number,
        )
        if question is None:
            break
        delta = Phi[question.a]-Phi[question.b]
        true_difference = float(delta @ truth.theta)
        if true_difference >= 0:
            winner, loser, response_delta = question.a, question.b, delta
        else:
            winner, loser, response_delta = question.b, question.a, -delta
        polytope.add_le(-response_delta, 0.0)
        asked.add(tuple(sorted((question.a, question.b))))
        if not polytope.contains(truth.theta):
            raise AssertionError("a true-consistent comparison removed the truth")
        polytope.feasible_point()
        trace.append(
            {
                "round": round_number,
                "a": question.a,
                "b": question.b,
                "winner": winner,
                "loser": loser,
                "lower": question.lower,
                "upper": question.upper,
                "width": question.width,
                "true_difference": true_difference,
            }
        )
    return IODInteraction(
        Phi=Phi,
        polytope=polytope,
        mandatory_questions=mandatory,
        pairwise_questions=len(trace),
        trace=trace,
        interaction_seconds=time.perf_counter()-started,
    )


_WORKER_PHI: Array | None = None
_WORKER_A: Array | None = None
_WORKER_B: Array | None = None
_WORKER_EPSILON: float = 0.0
_WORKER_ANCHOR: Array | None = None


def _init_membership_worker(
    Phi: Array,
    A: Array | None,
    b: Array | None,
    epsilon: float,
    feasible_anchor: Array,
) -> None:
    global _WORKER_PHI, _WORKER_A, _WORKER_B, _WORKER_EPSILON, _WORKER_ANCHOR
    _WORKER_PHI = np.asarray(Phi, dtype=np.float64)
    _WORKER_A = None if A is None else np.asarray(A, dtype=np.float64)
    _WORKER_B = None if b is None else np.asarray(b, dtype=np.float64)
    _WORKER_EPSILON = float(epsilon)
    _WORKER_ANCHOR = np.asarray(feasible_anchor, dtype=np.float64)


def _possible_worker(p: int) -> tuple[int, bool, Array | None, bool]:
    if _WORKER_PHI is None or _WORKER_ANCHOR is None:
        raise RuntimeError("membership worker was not initialized")
    Phi = _WORKER_PHI
    factor = 1.0+_WORKER_EPSILON
    # Deterministic constraint generation avoids building an n-by-K LP for
    # every tuple.  A witness is accepted only after separation against every
    # database row, so the final classification has the same all-q semantics.
    anchor_scores = Phi @ _WORKER_ANCHOR
    anchor_gaps = anchor_scores - factor * float(Phi[int(p)] @ _WORKER_ANCHOR)
    if float(np.max(anchor_gaps)) <= LP_TOLERANCE:
        return int(p), True, _WORKER_ANCHOR.copy(), False
    active = [int(np.argmax(anchor_scores))]
    active_set = set(active)
    for _ in range(len(Phi)+1):
        rows = Phi[np.asarray(active, dtype=int)]-factor*Phi[int(p)]
        A = rows if _WORKER_A is None else np.vstack([_WORKER_A, rows])
        zeros = np.zeros(len(rows), dtype=np.float64)
        b = zeros if _WORKER_B is None else np.concatenate([_WORKER_B, zeros])
        success, _, witness, status, message = _linprog(
            np.zeros(Phi.shape[1]), A, b, maximize=False
        )
        if not success:
            if status == 2:
                return int(p), False, None, False
            phase_status, phase_witness, phase_message = _phase_one_feasibility(
                A, b, Phi.shape[1]
            )
            if phase_status == "infeasible":
                return int(p), False, None, False
            if phase_status == "ambiguous" or phase_witness is None:
                # An unresolved tuple stays in the safe outer. This is the only
                # certificate-safe fallback: excluding it could create a false
                # negative. The ambiguity is counted and exported, never hidden.
                del message, phase_message
                return int(p), True, None, True
            witness = phase_witness
        assert witness is not None
        gaps = Phi @ witness-factor*float(Phi[int(p)] @ witness)
        q = int(np.argmax(gaps))
        if float(gaps[q]) <= LP_TOLERANCE:
            return int(p), True, witness, False
        if q in active_set:
            # The independent residual guard allows a slightly looser solver
            # tolerance.  A repeated active row in that narrow band is retained
            # in the safe outer rather than risking a false negative.
            if float(gaps[q]) <= 5e-7:
                return int(p), True, witness, False
            # Preserve outer-set safety if numerical separation cannot progress.
            return int(p), True, None, True
        active.append(q)
        active_set.add(q)
    # Exhaustion should be unreachable because every violated row is added at
    # most once. If numerical noise reaches it, retain the tuple in the safe
    # outer and report the unresolved classification instead of aborting.
    return int(p), True, None, True


_CORE_ANCHORS: Array | None = None
_CORE_ORDER: Array | None = None


def _init_core_worker(
    Phi: Array,
    A: Array | None,
    b: Array | None,
    epsilon: float,
    anchors: Array,
    competitor_order: Array,
) -> None:
    # Any feasible anchor is sufficient for shared worker initialization.
    _init_membership_worker(Phi, A, b, epsilon, anchors[0])
    global _CORE_ANCHORS, _CORE_ORDER
    _CORE_ANCHORS = np.asarray(anchors, dtype=np.float64)
    _CORE_ORDER = np.asarray(competitor_order, dtype=int)


def _core_worker(p: int) -> tuple[int, bool, bool]:
    if _WORKER_PHI is None or _CORE_ORDER is None:
        raise RuntimeError("core worker was not initialized")
    Phi = _WORKER_PHI
    for q in _CORE_ORDER:
        delta = Phi[int(q)]-(1.0+_WORKER_EPSILON)*Phi[int(p)]
        success, value, _, status, message = _linprog(
            delta, _WORKER_A, _WORKER_B, maximize=True
        )
        if not success or value is None:
            # A numerically unresolved candidate is excluded from the robust
            # core. Including it could create a false positive, so this is the
            # conservative direction. Exported diagnostics record the event.
            del status, message
            return int(p), False, True
        if value > LP_TOLERANCE:
            return int(p), False, False
    return int(p), True, False


def _pool_map(
    function: Any,
    items: Sequence[int],
    *,
    workers: int,
    initializer: Any,
    initargs: tuple[Any, ...],
) -> list[Any]:
    if workers <= 1 or len(items) < 32:
        initializer(*initargs)
        return [function(int(item)) for item in items]
    context = mp.get_context("fork")
    with context.Pool(processes=workers, initializer=initializer, initargs=initargs) as pool:
        return list(pool.imap(function, list(map(int, items)), chunksize=1))


@dataclass
class RelaxationSets:
    robust_core: Array
    safe_outer: Array
    possible_witnesses: dict[int, Array]
    classification_seconds: float
    ambiguous_outer_count: int = 0
    ambiguous_core_count: int = 0
    ambiguous_outer_indices: tuple[int, ...] = ()
    ambiguous_core_indices: tuple[int, ...] = ()


def compute_relaxation_sets(
    interaction: IODInteraction,
    epsilon: float,
    *,
    workers: int,
    compute_core: bool = True,
) -> RelaxationSets:
    """Classify existential outer and universal core for the LP relaxation."""

    started = time.perf_counter()
    Phi = interaction.Phi
    A, b = interaction.polytope.arrays()
    feasible_anchor = interaction.polytope.feasible_point()
    possible_rows = _pool_map(
        _possible_worker,
        list(range(len(Phi))),
        workers=workers,
        initializer=_init_membership_worker,
        initargs=(Phi, A, b, float(epsilon), feasible_anchor),
    )
    outer = np.asarray(
        [p for p, possible, _, _ in possible_rows if possible], dtype=int
    )
    witnesses = {
        int(p): np.asarray(witness, dtype=np.float64)
        for p, possible, witness, _ in possible_rows
        if possible and witness is not None
    }
    ambiguous_outer_indices = tuple(
        int(p) for p, _, _, ambiguous in possible_rows if ambiguous
    )
    ambiguous_outer_count = len(ambiguous_outer_indices)

    if not compute_core:
        if ambiguous_outer_count:
            warnings.warn(
                "HiGHS left "
                f"{ambiguous_outer_count} possible-membership classifications "
                "unresolved; certificate-safe conservative membership was used "
                "and exported.",
                RuntimeWarning,
            )
        return RelaxationSets(
            robust_core=np.empty(0, dtype=int),
            safe_outer=outer,
            possible_witnesses=witnesses,
            classification_seconds=time.perf_counter()-started,
            ambiguous_outer_count=ambiguous_outer_count,
            ambiguous_core_count=0,
            ambiguous_outer_indices=ambiguous_outer_indices,
            ambiguous_core_indices=(),
        )

    anchor_rows = [interaction.polytope.feasible_point(), *witnesses.values()]
    anchors = np.unique(np.round(np.vstack(anchor_rows), 12), axis=0)
    scores = anchors @ Phi.T
    best = scores.max(axis=1)
    violations = best[:, None]-(1.0+float(epsilon))*scores
    anchor_rejected = np.any(violations > LP_TOLERANCE, axis=0)
    survivors = np.flatnonzero(~anchor_rejected)

    # Try the strongest empirical competitors first; this normally rejects a
    # non-core row after very few LPs while preserving the all-competitor
    # semantics of the finite-atom LP relaxation if all n are needed.
    upper_competitor = np.max(scores, axis=0)
    base_order = np.argsort(-upper_competitor, kind="stable")
    competitor_order = base_order
    core_rows = _pool_map(
        _core_worker,
        survivors.tolist(),
        workers=workers,
        initializer=_init_core_worker,
        initargs=(Phi, A, b, float(epsilon), anchors, competitor_order),
    )
    core = np.asarray(
        [p for p, necessary, _ in core_rows if necessary], dtype=int
    )
    ambiguous_core_indices = tuple(
        int(p) for p, _, ambiguous in core_rows if ambiguous
    )
    ambiguous_core_count = len(ambiguous_core_indices)
    if ambiguous_outer_count or ambiguous_core_count:
        warnings.warn(
            "HiGHS left "
            f"{ambiguous_outer_count} possible-membership and "
            f"{ambiguous_core_count} robust-core classifications unresolved; "
            "certificate-safe conservative membership was used and exported.",
            RuntimeWarning,
        )
    return RelaxationSets(
        core,
        outer,
        witnesses,
        time.perf_counter()-started,
        ambiguous_outer_count,
        ambiguous_core_count,
        ambiguous_outer_indices,
        ambiguous_core_indices,
    )


# ---------------------------------------------------------------------------
# Original comparison family: linear internals, nonlinear oracle answers
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class NonlinearOracle:
    dictionary: tuple[float, ...]
    truth: TrueUtility
    mode: Literal["paper", "signed_monotone"]
    signed_shift: float

    def scores(self, X: Any) -> Array:
        return utility_scores(
            X,
            self.truth,
            self.dictionary,
            mode=self.mode,
            signed_shift=self.signed_shift,
        )

    def choose(self, X: Any) -> int:
        return int(np.argmax(self.scores(X)))


def top20_linear(X: Array, weights: Array, k: int = 20) -> Array:
    """Exactly k rows (or n if smaller), with a deterministic tie rule."""

    count = min(max(0, int(k)), len(X))
    scores = validate_normalized(X) @ np.asarray(weights, dtype=np.float64)
    return np.argsort(-scores, kind="stable")[:count]


def skyline_indices(X: Array, *, block_rows: int = 256) -> Array:
    values = validate_normalized(X)
    n = len(values)
    dominated = np.zeros(n, dtype=bool)
    for start in range(0, n, int(block_rows)):
        stop = min(n, start+int(block_rows))
        points = values[start:stop]
        local = np.zeros(len(points), dtype=bool)
        for other_start in range(0, n, int(block_rows)):
            others = values[other_start:min(n, other_start+int(block_rows))]
            local |= np.any(
                np.all(others[:, None, :] >= points[None, :, :], axis=2)
                & np.any(others[:, None, :] > points[None, :, :], axis=2),
                axis=0,
            )
        dominated[start:stop] = local
    return np.flatnonzero(~dominated)


def epsilon_front(X: Array, epsilon: float) -> Array:
    """Raw-coordinate ``(1+epsilon)`` front used by linear baselines.

    This rule is safe for the linear surrogate algorithms and for literal
    monomial IOD utilities whose active powers are all at least one.  It is
    *not* a safe IOD pre-pruner for concave powers below one; use
    :func:`iod_epsilon_front` for IOD.
    """

    values = validate_normalized(X)
    factor = 1.0+float(epsilon)
    if not np.isfinite(factor) or factor < 1.0:
        raise ValueError("epsilon must be finite and nonnegative")
    dominated = np.zeros(len(values), dtype=bool)
    for p in range(len(values)):
        candidates = np.all(values >= factor*values[p], axis=1) & np.any(
            values > factor*values[p], axis=1
        )
        candidates[p] = False
        dominated[p] = bool(np.any(candidates))
    return np.flatnonzero(~dominated)


def iod_epsilon_front(
    X: Array,
    epsilon: float,
    *,
    utility_mode: Literal["paper", "signed_monotone"],
    allowed_powers: Sequence[float],
) -> tuple[Array, dict[str, Any]]:
    """Return an IOD-safe front under the elicited exponent information.

    For the paper utility ``sum_i w_i*x_i**m_i`` with positive powers, raw
    domination by a factor ``r`` implies a utility improvement of at least
    ``r**m_min``.  Therefore a concave band needs

        r = (1 + epsilon)**(1/m_min),

    rather than the linear factor ``1 + epsilon``.  For powers at least one
    we deliberately retain the paper's more conservative raw factor so that
    previously completed non-concave cases remain bit-for-bit compatible.

    The signed monotone extension is not a homogeneous monomial family, so no
    raw multiplicative factor is certified for it.  Its safe fallback is to
    skip Stage-0 pruning and let the LP path see every row.
    """

    values = validate_normalized(X)
    epsilon = float(epsilon)
    if not np.isfinite(epsilon) or epsilon < 0.0:
        raise ValueError("epsilon must be finite and nonnegative")
    powers = np.asarray(allowed_powers, dtype=np.float64)
    if powers.ndim != 1 or len(powers) == 0 or not np.isfinite(powers).all():
        raise ValueError("allowed_powers must be a nonempty finite vector")

    if utility_mode == "signed_monotone":
        return np.arange(len(values), dtype=int), {
            "policy": IOD_PREPRUNE_POLICY,
            "status": "disabled_nonhomogeneous_signed_basis",
            "utility_epsilon": epsilon,
            "raw_epsilon": 0.0,
            "raw_factor": 1.0,
            "minimum_allowed_power": float(np.min(powers)),
        }
    if utility_mode != "paper":
        raise ValueError(utility_mode)
    if np.any(powers <= 0.0):
        raise ValueError("paper-mode allowed powers must be strictly positive")

    minimum_power = float(np.min(powers))
    utility_factor = 1.0 + epsilon
    if minimum_power < 1.0:
        # exp/log is more accurate than ** near epsilon=0.
        raw_factor = float(
            np.exp(np.log1p(epsilon) / minimum_power)
        )
        status = "concave_adjusted"
    else:
        raw_factor = utility_factor
        status = "paper_raw_factor_valid"
    raw_epsilon = raw_factor - 1.0
    front = epsilon_front(values, raw_epsilon)
    return front, {
        "policy": IOD_PREPRUNE_POLICY,
        "status": status,
        "utility_epsilon": epsilon,
        "raw_epsilon": raw_epsilon,
        "raw_factor": raw_factor,
        "minimum_allowed_power": minimum_power,
    }


def squeeze_u_linear(
    X: Array,
    oracle: NonlinearOracle,
    *,
    epsilon: float,
    question_budget: int,
    group_size: int | None = None,
    candidate_indices: Sequence[int] | None = None,
) -> tuple[Array, int]:
    values = validate_normalized(X)
    candidate_indices = (
        epsilon_front(values, epsilon)
        if candidate_indices is None
        else np.asarray(candidate_indices, dtype=int)
    )
    if len(candidate_indices) == 0:
        return np.empty(0, dtype=int), 0
    D = values[candidate_indices]
    minimum, maximum = D.min(axis=0), D.max(axis=0)
    d = values.shape[1]
    group_size = d if group_size is None else int(group_size)
    if group_size < 2:
        raise ValueError("Squeeze-u group_size must be at least two")
    artificial = np.repeat(minimum[None, :], d, axis=0)
    artificial[np.arange(d), np.arange(d)] = minimum+0.5*(maximum-minimum)
    questions = 0
    reference = 0
    i = 1
    while i < d and questions < int(question_budget):
        dimensions = [reference] + list(range(i, min(d, i+int(group_size)-1)))
        reference = int(dimensions[oracle.choose(artificial[dimensions])])
        i += int(group_size)-1
        questions += 1
    lower, upper = np.zeros(d), np.ones(d)
    lower[reference] = upper[reference] = 1.0
    current = 0
    while questions < int(question_budget):
        if current == reference:
            current = (current+1) % d
            continue
        grid = lower[current]+np.arange(int(group_size)+1)*(
            upper[current]-lower[current]
        )/float(group_size)
        queries = np.zeros((int(group_size), d), dtype=np.float64)
        for k in range(1, int(group_size)+1):
            if k <= int(group_size)-1:
                queries[k-1, reference] = grid[k:int(group_size)].sum()/float(group_size)
            queries[k-1, current] = k/float(group_size)
        chosen = oracle.choose(queries)+1
        lower[current], upper[current] = grid[chosen-1], grid[chosen]
        questions += 1
        current = (current+1) % d
    best_lower = float(np.max(D @ lower))
    keep = (1.0+float(epsilon))*(D @ upper) >= best_lower
    return candidate_indices[np.flatnonzero(keep)], questions


def _sample_feasible_linear_weights(
    d: int,
    constraints: Sequence[Array],
    count: int,
    *,
    rng: np.random.Generator,
    max_tries: int | None = None,
) -> Array:
    limit = int(max_tries if max_tries is not None else 10*int(count))
    matrix = np.vstack(constraints) if constraints else None
    accepted: list[Array] = []
    tries = 0
    while sum(len(block) for block in accepted) < int(count) and tries < limit:
        batch = min(64, limit-tries)
        samples = rng.dirichlet(np.ones(d), size=batch)
        feasible = (
            np.ones(batch, dtype=bool)
            if matrix is None
            else np.all(matrix @ samples.T >= 0.0, axis=0)
        )
        if np.any(feasible):
            accepted.append(samples[feasible])
        tries += batch
    if not accepted:
        return np.empty((0, d), dtype=np.float64)
    return np.vstack(accepted)[:int(count)]


def minr_mind_linear(
    X: Array,
    oracle: NonlinearOracle,
    *,
    epsilon: float,
    algorithm: Literal["MinR", "MinD"],
    question_budget: int,
    seed: int,
    group_size: int | None = None,
    candidate_groups: int = 10,
    weight_samples: int = 100,
    candidate_indices: Sequence[int] | None = None,
) -> tuple[Array, int]:
    """Possible-winner surrogate using a sampled feasible linear region."""

    if algorithm not in {"MinR", "MinD"}:
        raise ValueError(algorithm)
    values = validate_normalized(X)
    group_size = values.shape[1] if group_size is None else int(group_size)
    if group_size < 2:
        raise ValueError("MinR/MinD group_size must be at least two")
    candidates = (
        epsilon_front(values, epsilon)
        if candidate_indices is None
        else np.asarray(candidate_indices, dtype=int)
    )
    constraints: list[Array] = []
    rng = np.random.default_rng(int(seed))
    questions = 0
    while questions < int(question_budget) and len(candidates) >= int(group_size):
        best_group: Array | None = None
        best_metric: float | None = None
        for _ in range(int(candidate_groups)):
            group = (
                candidates.copy()
                if len(candidates) == int(group_size)
                else rng.choice(candidates, size=int(group_size), replace=False)
            )
            outcomes: list[float] = []
            for winner_position, winner_index in enumerate(group):
                temporary = list(constraints)
                winner = values[int(winner_index)]
                for other_position, other_index in enumerate(group):
                    if other_position != winner_position:
                        temporary.append(winner-values[int(other_index)])
                samples = _sample_feasible_linear_weights(
                    values.shape[1], temporary, int(weight_samples), rng=rng
                )
                if len(samples) == 0:
                    metric = math.inf
                elif algorithm == "MinR":
                    metric = float(np.max(samples.max(axis=0)-samples.min(axis=0)))
                else:
                    differences = samples[:, None, :]-samples[None, :, :]
                    metric = float(np.sqrt(np.max(np.sum(differences*differences, axis=2))))
                outcomes.append(metric)
            group_metric = float(np.mean(outcomes))
            if best_metric is None or group_metric < best_metric:
                best_metric, best_group = group_metric, np.asarray(group, dtype=int)
        if best_group is None:
            break
        selected = int(best_group[oracle.choose(values[best_group])])
        for other in best_group:
            if int(other) != selected:
                constraints.append(values[selected]-values[int(other)])
        questions += 1
        samples = _sample_feasible_linear_weights(
            values.shape[1], constraints, int(weight_samples), rng=rng
        )
        if len(samples):
            lower, upper = samples.min(axis=0), samples.max(axis=0)
            D = values[candidates]
            best_lower = float(np.max(D @ lower))
            candidates = candidates[(1.0+float(epsilon))*(D @ upper) >= best_lower]
    return np.asarray(candidates, dtype=int), questions


def set_metrics(
    output: Sequence[int],
    target: Sequence[int],
    *,
    true_scores: Sequence[float] | None = None,
    epsilon: float | None = None,
) -> dict[str, Any]:
    predicted, truth = set(map(int, output)), set(map(int, target))
    tp = len(predicted & truth)
    fp = len(predicted-truth)
    fn = len(truth-predicted)
    union = len(predicted | truth)
    metrics: dict[str, Any] = {
        "output_size": len(predicted),
        "target_size": len(truth),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": tp/len(predicted) if predicted else np.nan,
        "recall": tp/len(truth) if truth else 1.0,
        "f1": (2*tp)/(2*tp+fp+fn) if (2*tp+fp+fn) else 1.0,
        "jaccard": tp/union if union else 1.0,
        "symmetric_difference_error": (fp+fn)/len(truth) if truth else 0.0,
    }
    if true_scores is not None and epsilon is not None:
        scores = np.asarray(true_scores, dtype=np.float64)
        optimum = float(np.max(scores))
        false_positives = sorted(predicted-truth)
        if false_positives:
            spill = np.maximum(
                0.0,
                optimum-(1.0+float(epsilon))*scores[false_positives],
            )
            alpha_absolute = float(np.max(spill))
        else:
            alpha_absolute = 0.0
        metrics.update(
            {
                # Definition 4 additionally requires no false negatives.  A
                # baseline with fn>0 is not an alpha-approximation even though
                # its false-positive spill can still be reported diagnostically.
                "alpha_approximation_valid": bool(fn == 0),
                "alpha_spill_absolute": alpha_absolute,
                "alpha_spill_relative": (
                    alpha_absolute/optimum if optimum > 0 else 0.0
                ),
            }
        )
    return metrics


def _method_row(
    *,
    protocol: str,
    dataset: str,
    replicate: int,
    seed: int,
    n: int,
    d: int,
    epsilon: float,
    method: str,
    output: Sequence[int],
    target: Sequence[int],
    runtime_seconds: float,
    pairwise_questions: int,
    mandatory_questions: int,
    truth: TrueUtility,
    config: ExperimentConfig,
    true_scores: Sequence[float] | None = None,
    total_budget_cap: int | None = None,
) -> dict[str, Any]:
    is_iod = method == METHOD_ORDER[-1]
    is_multiway_baseline = method in METHOD_ORDER[2:5]
    return {
        "protocol": protocol,
        "dataset": dataset,
        "replicate": int(replicate),
        "seed": int(seed),
        "n": int(n),
        "d": int(d),
        "epsilon": float(epsilon),
        "method": method,
        "pairwise_questions": int(pairwise_questions),
        "adaptive_binary_prompts": int(pairwise_questions) if is_iod else 0,
        "multiway_choice_prompts": int(pairwise_questions) if is_multiway_baseline else 0,
        "multiway_choice_arity": int(d) if is_multiway_baseline else np.nan,
        "swing_prompts": int(d+1) if is_iod else 0,
        "exponent_range_prompts": 1 if is_iod else 0,
        "comparison_options_shown": (
            int(2*pairwise_questions)
            if is_iod
            else int(d*pairwise_questions) if is_multiway_baseline else 0
        ),
        "mandatory_questions": int(mandatory_questions),
        "total_questions": int(pairwise_questions+mandatory_questions),
        "total_budget_cap": np.nan if total_budget_cap is None else int(total_budget_cap),
        "runtime_seconds": float(runtime_seconds),
        "utility_mode": config.utility_mode,
        "power_dictionary": json.dumps(list(map(float, config.powers))),
        "power_sampling_policy": POWER_SAMPLING_POLICY,
        "exponent_range_policy": config.exponent_range_policy,
        "selected_power_band": json.dumps(
            []
            if truth.selected_power_band is None
            else truth.selected_power_band.tolist()
        ),
        "true_weights": json.dumps(truth.weights.tolist()),
        "true_powers": json.dumps(truth.powers.tolist()),
        "true_power_min": float(np.min(truth.powers)),
        "true_power_max": float(np.max(truth.powers)),
        "true_power_span": float(np.ptp(truth.powers)),
        "true_power_max_allowed_gap": float(config.truth_power_max_gap),
        "signed_shift": config.signed_shift if config.utility_mode == "signed_monotone" else np.nan,
        **set_metrics(
            output,
            target,
            true_scores=true_scores,
            epsilon=epsilon,
        ),
    }


def run_shared_case(
    X: Array,
    *,
    dataset: str,
    replicate: int,
    seed: int,
    config: ExperimentConfig,
    epsilons: Sequence[float],
    iod_adaptive_questions: int,
    baseline_questions: int,
    protocol: str,
    total_budget_cap: int | None = None,
    forced_power_band: Sequence[float] | None = None,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Run every original-paper method on identical case inputs.

    IOD is rebuilt for each epsilon because Algorithm 1 performs the
    ``(1+epsilon)`` pre-prune before selecting its adaptive questions.  The V2
    implementation incorrectly reused one full-database interaction across
    all epsilon values.
    """

    values = validate_normalized(X)
    truth = make_true_utility(
        values.shape[1],
        config.powers,
        seed=seed,
        max_power_gap=config.truth_power_max_gap,
        forced_band=forced_power_band,
    )
    oracle = NonlinearOracle(config.powers, truth, config.utility_mode, config.signed_shift)
    true_scores = oracle.scores(values)
    rows: list[dict[str, Any]] = []

    started = time.perf_counter()
    top20_output = top20_linear(values, truth.weights)
    top20_seconds = time.perf_counter()-started
    started = time.perf_counter()
    skyline_output = skyline_indices(values)
    skyline_seconds = time.perf_counter()-started

    memberships: dict[str, Any] = {
        "truth": {
            "weights": truth.weights.tolist(),
            "powers": truth.powers.tolist(),
            "power_indices": truth.power_indices.tolist(),
            "selected_power_band": (
                []
                if truth.selected_power_band is None
                else truth.selected_power_band.tolist()
            ),
            "power_sampling_policy": POWER_SAMPLING_POLICY,
            "power_span": float(np.ptp(truth.powers)),
            "max_allowed_power_gap": float(config.truth_power_max_gap),
            "exponent_range_policy": config.exponent_range_policy,
        },
        "epsilons": {},
    }
    for epsilon in epsilons:
        epsilon = float(epsilon)
        target = target_indices(true_scores, epsilon)

        allowed_iod_powers = (
            np.asarray(truth.selected_power_band, dtype=np.float64)
            if config.exponent_range_policy == "elicited_band"
            else np.asarray(config.powers, dtype=np.float64)
        )
        iod_preprune_started = time.perf_counter()
        iod_candidate_indices, iod_preprune_diagnostics = iod_epsilon_front(
            values,
            epsilon,
            utility_mode=config.utility_mode,
            allowed_powers=allowed_iod_powers,
        )
        iod_preprune_seconds = time.perf_counter()-iod_preprune_started
        candidate_values = values[iod_candidate_indices]
        missing_from_iod_front = np.setdiff1d(
            target, iod_candidate_indices, assume_unique=False
        )
        if len(missing_from_iod_front):
            raise AssertionError(
                "IOD power-safe front removed true targets: "
                f"epsilon={epsilon}, mode={config.utility_mode}, "
                f"allowed_powers={allowed_iod_powers.tolist()}, "
                f"raw_factor={iod_preprune_diagnostics['raw_factor']}, "
                f"missing={missing_from_iod_front.tolist()}"
            )

        # The comparison algorithms are linear surrogates and retain their
        # own raw-coordinate front.  They must not receive IOD's elicited
        # exponent band as side information.
        baseline_preprune_started = time.perf_counter()
        baseline_candidate_indices = epsilon_front(values, epsilon)
        baseline_preprune_seconds = time.perf_counter()-baseline_preprune_started

        interaction = run_iod_interaction(
            candidate_values,
            truth,
            config,
            adaptive_questions=iod_adaptive_questions,
            seed=int(seed)+int(round(1_000_000*epsilon)),
        )
        relaxation = compute_relaxation_sets(
            interaction, epsilon, workers=config.workers, compute_core=False
        )
        safe_outer = iod_candidate_indices[relaxation.safe_outer]
        target_set = set(map(int, target))
        if not target_set <= set(map(int, safe_outer)):
            raise AssertionError("safe outer missed a true target")

        outputs: dict[str, tuple[Array, int, int, float]] = {}
        algorithm_seconds: dict[str, float] = {}
        method_preprune_seconds: dict[str, float] = {}
        outputs[METHOD_ORDER[0]] = (top20_output, 0, 0, top20_seconds)
        outputs[METHOD_ORDER[1]] = (skyline_output, 0, 0, skyline_seconds)
        algorithm_seconds[METHOD_ORDER[0]] = top20_seconds
        algorithm_seconds[METHOD_ORDER[1]] = skyline_seconds
        method_preprune_seconds[METHOD_ORDER[0]] = 0.0
        method_preprune_seconds[METHOD_ORDER[1]] = 0.0

        started = time.perf_counter()
        squeeze, squeeze_q = squeeze_u_linear(
            values,
            oracle,
            epsilon=epsilon,
            question_budget=baseline_questions,
            candidate_indices=baseline_candidate_indices,
        )
        squeeze_seconds = time.perf_counter()-started
        outputs[METHOD_ORDER[2]] = (
            squeeze,
            squeeze_q,
            0,
            baseline_preprune_seconds+squeeze_seconds,
        )
        algorithm_seconds[METHOD_ORDER[2]] = squeeze_seconds
        method_preprune_seconds[METHOD_ORDER[2]] = baseline_preprune_seconds
        for offset, algorithm, method in ((101, "MinR", METHOD_ORDER[3]), (102, "MinD", METHOD_ORDER[4])):
            started = time.perf_counter()
            output, questions = minr_mind_linear(
                values,
                oracle,
                epsilon=epsilon,
                algorithm=algorithm,  # type: ignore[arg-type]
                question_budget=baseline_questions,
                seed=int(seed)+offset,
                candidate_indices=baseline_candidate_indices,
            )
            method_seconds = time.perf_counter()-started
            outputs[method] = (
                output,
                questions,
                0,
                baseline_preprune_seconds+method_seconds,
            )
            algorithm_seconds[method] = method_seconds
            method_preprune_seconds[method] = baseline_preprune_seconds
        joint_iod_seconds = (
            iod_preprune_seconds
            + interaction.interaction_seconds
            + relaxation.classification_seconds
        )
        outputs[METHOD_ORDER[5]] = (
            safe_outer,
            interaction.pairwise_questions,
            interaction.mandatory_questions,
            joint_iod_seconds,
        )
        algorithm_seconds[METHOD_ORDER[-1]] = (
            interaction.interaction_seconds+relaxation.classification_seconds
        )
        method_preprune_seconds[METHOD_ORDER[-1]] = iod_preprune_seconds
        if tuple(outputs) != METHOD_ORDER:
            raise AssertionError("method order/contract changed")
        for method, (output, pairwise, mandatory, runtime) in outputs.items():
            record = _method_row(
                protocol=protocol,
                dataset=dataset,
                replicate=replicate,
                seed=seed,
                n=len(values),
                d=values.shape[1],
                epsilon=epsilon,
                method=method,
                output=output,
                target=target,
                runtime_seconds=runtime,
                pairwise_questions=pairwise,
                mandatory_questions=mandatory,
                truth=truth,
                config=config,
                true_scores=true_scores,
                total_budget_cap=total_budget_cap,
            )
            record["candidate_front_size"] = int(
                len(iod_candidate_indices)
                if method == METHOD_ORDER[-1]
                else len(baseline_candidate_indices)
                if method in METHOD_ORDER[2:5]
                else len(values)
            )
            record["candidate_front_policy"] = (
                str(iod_preprune_diagnostics["status"])
                if method == METHOD_ORDER[-1]
                else "raw_linear_surrogate"
                if method in METHOD_ORDER[2:5]
                else "not_applicable"
            )
            record["preprune_seconds"] = float(method_preprune_seconds[method])
            record["algorithm_seconds"] = float(algorithm_seconds[method])
            record["classification_status"] = "full_finite_lp"
            if method == METHOD_ORDER[-1]:
                record["lp_ambiguous_outer_count"] = int(
                    relaxation.ambiguous_outer_count
                )
            else:
                record["lp_ambiguous_outer_count"] = np.nan
            rows.append(record)
        memberships["epsilons"][str(epsilon)] = {
            "target": target.tolist(),
            "candidate_front": iod_candidate_indices.tolist(),
            "iod_candidate_front": iod_candidate_indices.tolist(),
            "baseline_candidate_front": baseline_candidate_indices.tolist(),
            "iod_preprune_diagnostics": iod_preprune_diagnostics,
            "interaction_trace": interaction.trace,
            "outputs": {
                method: output.tolist()
                for method, (output, _, _, _) in outputs.items()
            },
            "lp_diagnostics": {
                "ambiguous_outer_count": int(relaxation.ambiguous_outer_count),
                "ambiguous_outer_indices": list(relaxation.ambiguous_outer_indices),
                "fallback_policy": (
                    "unresolved possible membership is retained in the safe outer"
                ),
            },
        }
    return pd.DataFrame(rows), memberships


# ---------------------------------------------------------------------------
# Configuration-hashed, completed-case-only local checkpoints
# ---------------------------------------------------------------------------


class CompletedCaseCache:
    def __init__(self, root: Path, config_hash: str) -> None:
        self.root = Path(root) / "checkpoints" / config_hash
        self.root.mkdir(parents=True, exist_ok=True)
        self.config_hash = str(config_hash)

    @staticmethod
    def _slug(case_id: str) -> str:
        clean = "".join(character if character.isalnum() or character in "-_" else "_" for character in case_id)
        return clean[:180]

    def load(
        self,
        case_id: str,
        *,
        expected_epsilons: Sequence[float],
        required_case_revision: str | None = None,
    ) -> tuple[pd.DataFrame, dict[str, Any]] | None:
        directory = self.root / self._slug(case_id)
        marker = directory / ".complete.json"
        rows_path = directory / "rows.csv"
        memberships_path = directory / "memberships.json"
        if not marker.exists() or not rows_path.exists() or not memberships_path.exists():
            return None
        try:
            metadata = json.loads(marker.read_text(encoding="utf-8"))
            if (
                metadata.get("cache_schema") != CACHE_SCHEMA
                or metadata.get("config_hash") != self.config_hash
                or metadata.get("case_id") != case_id
            ):
                return None
            if (
                required_case_revision is not None
                and metadata.get("case_revision") != required_case_revision
            ):
                return None
            rows = pd.read_csv(rows_path)
            memberships = json.loads(memberships_path.read_text(encoding="utf-8"))
        except (OSError, ValueError, json.JSONDecodeError):
            return None
        if not {"epsilon", "method"}.issubset(rows.columns):
            return None
        if not isinstance(memberships, dict):
            return None
        expected = {float(value) for value in expected_epsilons}
        if set(map(float, rows["epsilon"].unique())) != expected:
            return None
        for epsilon in expected:
            block = rows[np.isclose(rows["epsilon"], epsilon)]
            if tuple(block["method"].tolist()) != METHOD_ORDER:
                return None
        return rows, memberships

    def commit(
        self,
        case_id: str,
        rows: pd.DataFrame,
        memberships: Mapping[str, Any],
        *,
        case_revision: str | None = None,
    ) -> None:
        """Commit one whole successful case; the completion marker is last."""

        directory = self.root / self._slug(case_id)
        directory.mkdir(parents=True, exist_ok=True)
        atomic_csv(directory / "rows.csv", rows)
        atomic_json(directory / "memberships.json", memberships)
        atomic_json(
            directory / ".complete.json",
            {
                "cache_schema": CACHE_SCHEMA,
                "config_hash": self.config_hash,
                "case_id": case_id,
                "case_revision": case_revision,
                "rows": len(rows),
                "completed_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            },
        )


def run_or_resume_case(
    cache: CompletedCaseCache,
    case_id: str,
    *,
    expected_epsilons: Sequence[float],
    run: Any,
    case_revision: str | None = None,
) -> tuple[pd.DataFrame, dict[str, Any], bool]:
    cached = cache.load(
        case_id,
        expected_epsilons=expected_epsilons,
        required_case_revision=case_revision,
    )
    if cached is not None:
        return cached[0], cached[1], True
    rows, memberships = run()
    cache.commit(
        case_id,
        rows,
        memberships,
        case_revision=case_revision,
    )
    return rows, memberships, False


# ---------------------------------------------------------------------------
# Paper-facing aggregation and exports
# ---------------------------------------------------------------------------


def legacy_summary_results(detailed: pd.DataFrame) -> pd.DataFrame:
    # The matched-budget protocol contains several distinct total caps.  Keep
    # that cap in the aggregation key; otherwise Q=12, Q=20, and Q=25 would be
    # averaged into one misleading row.  Older/imported detailed tables may
    # not have the column, in which case they form the usual single NaN group.
    detailed = detailed.copy()
    if "total_budget_cap" not in detailed.columns:
        detailed["total_budget_cap"] = np.nan
    group_columns = [
        "protocol", "dataset", "n", "d", "epsilon", "total_budget_cap", "method"
    ]
    rows: list[dict[str, Any]] = []
    for keys, group in detailed.groupby(group_columns, sort=False, dropna=False):
        row = dict(zip(group_columns, keys))
        row.update(
            {
                "runs": len(group),
                "output_size_mean": float(group["output_size"].mean()),
                "target_size_mean": float(group["target_size"].mean()),
                "precision_mean": float(group["precision"].mean()) if group["precision"].notna().any() else np.nan,
                "recall_mean": float(group["recall"].mean()),
                "symmetric_difference_error_mean": float(
                    group["symmetric_difference_error"].mean()
                ),
                "pairwise_questions_mean": float(group["pairwise_questions"].mean()),
                "mandatory_questions_mean": float(group["mandatory_questions"].mean()),
                "total_questions_mean": float(group["total_questions"].mean()),
                "runtime_seconds_median": float(group["runtime_seconds"].median()),
            }
        )
        rows.append(row)
    summary = pd.DataFrame(rows)
    if len(summary):
        rank = {method: index for index, method in enumerate(METHOD_ORDER)}
        summary["_rank"] = summary["method"].map(rank)
        summary = summary.sort_values(
            ["protocol", "dataset", "d", "epsilon", "total_budget_cap", "_rank"],
            kind="stable",
        ).drop(columns="_rank")
    return summary.reset_index(drop=True)


def _latex_escape(text: str) -> str:
    return str(text).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")


def summary_to_latex(summary: pd.DataFrame, *, caption: str, label: str) -> str:
    include_budget_cap = (
        "total_budget_cap" in summary.columns
        and summary["total_budget_cap"].notna().any()
    )
    columns = ["dataset", "d", "epsilon"]
    if include_budget_cap:
        columns.append("total_budget_cap")
    columns += [
        "method", "output_size_mean",
        "precision_mean", "recall_mean", "symmetric_difference_error_mean",
        "pairwise_questions_mean", "mandatory_questions_mean",
        "total_questions_mean", "runtime_seconds_median",
    ]
    data = summary[columns].copy()
    headers = ["Data", "$d$", "$\\varepsilon$"]
    if include_budget_cap:
        headers.append("$Q_{max}$")
    headers += [
        "Method/output", "$|S|$", "Prec.",
        "Rec.", "$E_{\\triangle}$", "$q_{pair}$", "$q_{init}$", "$q_{total}$", "Time (s)",
    ]
    column_spec = "lrr" + ("r" if include_budget_cap else "") + "l" + "r"*8
    lines = [
        r"\begin{table*}[t]", r"\centering", r"\scriptsize",
        f"\\caption{{{caption}}}", f"\\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%", f"\\begin{{tabular}}{{{column_spec}}}",
        r"\toprule", " & ".join(headers)+r" \\", r"\midrule",
    ]
    for record in data.to_dict(orient="records"):
        values = [
            _latex_escape(record["dataset"]),
            str(int(record["d"])),
            f"{float(record['epsilon']):.2f}",
        ]
        if include_budget_cap:
            values.append(str(int(record["total_budget_cap"])))
        values += [
            _latex_escape(record["method"]),
            f"{float(record['output_size_mean']):.1f}",
            "--" if pd.isna(record["precision_mean"]) else f"{float(record['precision_mean']):.3f}",
            f"{float(record['recall_mean']):.3f}",
            f"{float(record['symmetric_difference_error_mean']):.3f}",
            f"{float(record['pairwise_questions_mean']):.1f}",
            f"{float(record['mandatory_questions_mean']):.1f}",
            f"{float(record['total_questions_mean']):.1f}",
            f"{float(record['runtime_seconds_median']):.3f}",
        ]
        lines.append(" & ".join(values)+r" \\")
    lines.extend(
        [
            r"\bottomrule", r"\end{tabular}}", r"\vspace{1mm}",
            r"\parbox{0.98\textwidth}{\footnotesize $E_{\triangle}=(FP+FN)/|I|$. "
            r"The comparison uses IOD's formal safe outer set. "
            r"Squeeze-$u$ is a linear surrogate; MinR and MinD use sampled feasible linear weights. "
            r"Prompt counts do not represent equal respondent effort across different prompt formats.}",
            r"\end{table*}", "",
        ]
    )
    return "\n".join(lines)


def run_experiment_suite(config: ExperimentConfig) -> dict[str, Any]:
    config.validate()
    workbook = _find_workbook(Path(config.workbook_path))
    workbook_digest = file_sha256(workbook)
    config_hash = stable_hash(config.hash_payload(workbook_sha256=workbook_digest))
    root = Path(config.result_root)
    run_dir = root / f"run_{config_hash[:12]}"
    run_dir.mkdir(parents=True, exist_ok=True)
    cache = CompletedCaseCache(root, config_hash)
    print(
        f"IOD nonlinear CPU v3.1 | build={RELEASE_BUILD} | "
        f"profile={config.profile} | mode={config.utility_mode} | "
        f"powers={config.powers} | exponent_range={config.exponent_range_policy} | "
        f"true_power_max_gap={config.truth_power_max_gap:g} | "
        f"workers={config.workers} | quick={config.quick} | hash={config_hash[:12]} | "
        f"iod_preprune={IOD_PREPRUNE_POLICY} | "
        f"concave_case_revision={CONCAVE_CASE_REVISION}",
        flush=True,
    )

    X_real, raw_real, processed_real = load_real_estate(workbook)
    atomic_csv(run_dir / "real_input_raw.csv", raw_real)
    atomic_csv(run_dir / "real_input_processed.csv", processed_real)
    all_frames: list[pd.DataFrame] = []
    case_manifest: list[dict[str, Any]] = []
    audit_membership_dir = run_dir / "audit" / "memberships"
    audit_membership_dir.mkdir(parents=True, exist_ok=True)

    def retain_case_audit(
        case_id: str,
        frame: pd.DataFrame,
        membership: Mapping[str, Any],
        *,
        dataset_hash: str,
        resumed: bool,
    ) -> None:
        frame["dataset_sha256"] = str(dataset_hash)
        atomic_json(audit_membership_dir / f"{case_id}.json", membership)
        case_manifest.append(
            {
                "case_id": case_id,
                "resumed": bool(resumed),
                "rows": int(len(frame)),
                "dataset_sha256": str(dataset_hash),
            }
        )

    for replicate, root_seed in enumerate(config.seeds):
        for d in config.dimensions:
            X = generate_anti_correlated(
                config.synthetic_n, int(d), seed=int(root_seed)+31*int(d)
            )
            if replicate == 0:
                atomic_csv(
                    run_dir / f"synthetic_n{config.synthetic_n}_d{d}.csv",
                    pd.DataFrame(X, columns=[f"x{i+1}" for i in range(int(d))]),
                )
            utility_seed = int(root_seed)+1009*int(d)
            case_id = f"paper_synthetic_r{replicate}_d{d}"
            frame, membership, resumed = run_or_resume_case(
                cache,
                case_id,
                expected_epsilons=config.epsilons,
                run=lambda X=X, d=d, utility_seed=utility_seed: run_shared_case(
                    X,
                    dataset="anti-correlated synthetic",
                    replicate=replicate,
                    seed=utility_seed,
                    config=config,
                    epsilons=config.epsilons,
                    iod_adaptive_questions=config.adaptive_pairwise_questions,
                    baseline_questions=config.baseline_pairwise_questions,
                    protocol="paper_protocol",
                ),
            )
            dataset_hash = hashlib.sha256(
                np.ascontiguousarray(X, dtype=np.float64).tobytes()
            ).hexdigest()
            frame["power_regime"] = "random_elicited_band"
            retain_case_audit(
                case_id,
                frame,
                membership,
                dataset_hash=dataset_hash,
                resumed=resumed,
            )
            all_frames.append(frame)
            print(f"completed {case_id} (resumed={resumed})", flush=True)

        real_seed = int(root_seed)+90001
        case_id = f"paper_real_r{replicate}"
        frame, membership, resumed = run_or_resume_case(
            cache,
            case_id,
            expected_epsilons=config.epsilons,
            run=lambda real_seed=real_seed: run_shared_case(
                X_real,
                dataset="UCI Real Estate Valuation (414 rows)",
                replicate=replicate,
                seed=real_seed,
                config=config,
                epsilons=config.epsilons,
                iod_adaptive_questions=config.adaptive_pairwise_questions,
                baseline_questions=config.baseline_pairwise_questions,
                protocol="paper_protocol",
            ),
        )
        real_array_hash = hashlib.sha256(
            np.ascontiguousarray(X_real, dtype=np.float64).tobytes()
        ).hexdigest()
        frame["power_regime"] = "random_elicited_band"
        retain_case_audit(
            case_id,
            frame,
            membership,
            dataset_hash=real_array_hash,
            resumed=resumed,
        )
        all_frames.append(frame)
        print(f"completed {case_id} (resumed={resumed})", flush=True)

        d = int(config.matched_budget_dimension)
        X_budget = generate_anti_correlated(
            config.synthetic_n, d, seed=int(root_seed)+31*d
        )
        utility_seed = int(root_seed)+1009*d
        mandatory = d+2
        for total_cap in config.matched_total_budgets:
            adaptive = max(0, int(total_cap)-mandatory)
            case_id = f"budget_r{replicate}_Q{total_cap}"
            frame, membership, resumed = run_or_resume_case(
                cache,
                case_id,
                expected_epsilons=(config.matched_budget_epsilon,),
                run=lambda adaptive=adaptive, total_cap=total_cap: run_shared_case(
                    X_budget,
                    dataset="anti-correlated synthetic",
                    replicate=replicate,
                    seed=utility_seed,
                    config=config,
                    epsilons=(config.matched_budget_epsilon,),
                    iod_adaptive_questions=adaptive,
                    baseline_questions=int(total_cap),
                    protocol="matched_total_budget",
                    total_budget_cap=int(total_cap),
                ),
            )
            budget_hash = hashlib.sha256(
                np.ascontiguousarray(X_budget, dtype=np.float64).tobytes()
            ).hexdigest()
            frame["power_regime"] = "random_elicited_band"
            retain_case_audit(
                case_id,
                frame,
                membership,
                dataset_hash=budget_hash,
                resumed=resumed,
            )
            # Enforce the total-budget contract for both IOD rows.
            iod_rows = frame[frame["method"].eq(METHOD_ORDER[-1])]
            if (iod_rows["total_questions"] > int(total_cap)).any():
                raise AssertionError("IOD exceeded the matched total budget")
            all_frames.append(frame)
            print(f"completed {case_id} (resumed={resumed})", flush=True)

        # Direct matched-total comparisons at every paper dimension.  IOD gets
        # d+2 initialization prompts plus ten binary refinements; each linear
        # interactive baseline gets the same numeric total as multiway prompts.
        for matched_d in config.dimensions:
            matched_d = int(matched_d)
            matched_cap = matched_d+2+int(config.adaptive_pairwise_questions)
            X_matched = generate_anti_correlated(
                config.synthetic_n,
                matched_d,
                seed=int(root_seed)+31*matched_d,
            )
            matched_seed = int(root_seed)+1009*matched_d
            case_id = (
                f"matched_dimension_r{replicate}_d{matched_d}_Q{matched_cap}"
            )
            frame, membership, resumed = run_or_resume_case(
                cache,
                case_id,
                expected_epsilons=config.epsilons,
                run=lambda X_matched=X_matched, matched_d=matched_d,
                matched_seed=matched_seed, matched_cap=matched_cap: run_shared_case(
                    X_matched,
                    dataset="anti-correlated synthetic",
                    replicate=replicate,
                    seed=matched_seed,
                    config=config,
                    epsilons=config.epsilons,
                    iod_adaptive_questions=config.adaptive_pairwise_questions,
                    baseline_questions=matched_cap,
                    protocol="matched_total_by_dimension",
                    total_budget_cap=matched_cap,
                ),
            )
            frame["power_regime"] = "random_elicited_band"
            matched_hash = hashlib.sha256(
                np.ascontiguousarray(X_matched, dtype=np.float64).tobytes()
            ).hexdigest()
            retain_case_audit(
                case_id,
                frame,
                membership,
                dataset_hash=matched_hash,
                resumed=resumed,
            )
            all_frames.append(frame)
            print(f"completed {case_id} (resumed={resumed})", flush=True)

        real_cap = X_real.shape[1]+2+int(config.adaptive_pairwise_questions)
        case_id = f"matched_real_r{replicate}_Q{real_cap}"
        frame, membership, resumed = run_or_resume_case(
            cache,
            case_id,
            expected_epsilons=config.epsilons,
            run=lambda real_seed=real_seed, real_cap=real_cap: run_shared_case(
                X_real,
                dataset="UCI Real Estate Valuation (414 rows)",
                replicate=replicate,
                seed=real_seed,
                config=config,
                epsilons=config.epsilons,
                iod_adaptive_questions=config.adaptive_pairwise_questions,
                baseline_questions=real_cap,
                protocol="matched_total_by_dimension",
                total_budget_cap=real_cap,
            ),
        )
        frame["power_regime"] = "random_elicited_band"
        retain_case_audit(
            case_id,
            frame,
            membership,
            dataset_hash=real_array_hash,
            resumed=resumed,
        )
        all_frames.append(frame)
        print(f"completed {case_id} (resumed={resumed})", flush=True)

        if (
            "power_band_ablation" in config.studies
            and not config.quick
            and config.utility_mode == "paper"
        ):
            ablation_X = generate_anti_correlated(
                config.synthetic_n, 5, seed=int(root_seed)+31*5
            )
            ablation_seed = int(root_seed)+1009*5
            ablation_specs = (
                (
                    "linear_control",
                    replace(
                        config,
                        powers=(1.0,),
                        truth_power_max_gap=0.0,
                        exponent_range_policy="elicited_band",
                    ),
                    (1.0,),
                ),
                (
                    "concave_025_050_075",
                    replace(
                        config,
                        powers=(0.25, 0.50, 0.75),
                        truth_power_max_gap=0.50,
                        exponent_range_policy="elicited_band",
                    ),
                    (0.25, 0.50, 0.75),
                ),
                ("convex_low_1_2_3", config, (1.0, 2.0, 3.0)),
                ("convex_mid_4_5_6", config, (4.0, 5.0, 6.0)),
                ("convex_high_8_9_10", config, (8.0, 9.0, 10.0)),
                (
                    "full_dictionary_stress",
                    replace(
                        config,
                        exponent_range_policy="full_dictionary_stress",
                    ),
                    None,
                ),
            )
            for regime, regime_config, forced_band in ablation_specs:
                case_id = f"power_ablation_r{replicate}_{regime}"
                frame, membership, resumed = run_or_resume_case(
                    cache,
                    case_id,
                    expected_epsilons=(0.05,),
                    case_revision=(
                        CONCAVE_CASE_REVISION
                        if regime == "concave_025_050_075"
                        else None
                    ),
                    run=lambda regime_config=regime_config,
                    forced_band=forced_band: run_shared_case(
                        ablation_X,
                        dataset="anti-correlated synthetic",
                        replicate=replicate,
                        seed=ablation_seed,
                        config=regime_config,
                        epsilons=(0.05,),
                        iod_adaptive_questions=config.adaptive_pairwise_questions,
                        baseline_questions=config.baseline_pairwise_questions,
                        protocol="power_band_ablation",
                        forced_power_band=forced_band,
                    ),
                )
                frame["power_regime"] = regime
                ablation_hash = hashlib.sha256(
                    np.ascontiguousarray(ablation_X, dtype=np.float64).tobytes()
                ).hexdigest()
                retain_case_audit(
                    case_id,
                    frame,
                    membership,
                    dataset_hash=ablation_hash,
                    resumed=resumed,
                )
                all_frames.append(frame)
                print(f"completed {case_id} (resumed={resumed})", flush=True)

    scalability_result = None
    if config.run_scalability and "scalability" in config.studies:
        bindings = ExperimentBindings(
            generate_anti_correlated=generate_anti_correlated,
            make_true_utility=make_true_utility,
            nonlinear_oracle_type=NonlinearOracle,
            run_iod_interaction=run_iod_interaction,
            compute_relaxation_sets=compute_relaxation_sets,
            target_indices=target_indices,
            set_metrics=set_metrics,
        )
        scalability_result = run_scalability_suite(
            config,
            bindings,
            sizes=config.scalability_sizes,
            seed=int(config.seeds[0]) + 700_001,
            dimension=config.scalability_dimension,
            epsilon=config.scalability_epsilon,
            full_classification_max_n=config.scalability_full_lp_max_n,
            large_n_exact_candidates_per_set=(
                config.scalability_large_n_exact_candidates_per_set
            ),
            iod_time_limit_seconds=config.scalability_exact_refinement_seconds,
            methods=METHOD_ORDER,
            include_memberships=True,
        )
        scalability_rows = scalability_result.rows.copy()
        scalability_rows["dataset"] = "anti-correlated synthetic scalability"
        scalability_rows["replicate"] = 0
        scalability_rows["power_regime"] = "random_elicited_band"
        scalability_rows["exponent_range_policy"] = config.exponent_range_policy
        scalability_rows["utility_mode"] = config.utility_mode
        scalability_rows["true_power_max_allowed_gap"] = config.truth_power_max_gap
        all_frames.append(scalability_rows)
        atomic_csv(run_dir / "scalability_all_algorithms.csv", scalability_rows)
        atomic_json(
            run_dir / "audit" / "scalability_memberships_and_bounds.json",
            scalability_result.memberships,
        )
        for n in config.scalability_sizes:
            block = scalability_rows[scalability_rows["n"].eq(int(n))]
            case_manifest.append(
                {
                    "case_id": f"scalability_n{int(n)}",
                    "resumed": False,
                    "rows": int(len(block)),
                    "classification_complete": bool(
                        block["classification_complete"].all()
                    ),
                    "classification_status": sorted(
                        set(map(str, block["classification_status"]))
                    ),
                }
            )
        print("completed scalability bank", flush=True)

    detailed = pd.concat(all_frames, ignore_index=True)
    summary = legacy_summary_results(detailed)
    paper_detailed = detailed[detailed["protocol"] == "paper_protocol"].reset_index(drop=True)
    budget_detailed = detailed[detailed["protocol"] == "matched_total_budget"].reset_index(drop=True)
    paper_summary = summary[summary["protocol"] == "paper_protocol"].reset_index(drop=True)
    budget_summary = summary[summary["protocol"] == "matched_total_budget"].reset_index(drop=True)
    atomic_csv(run_dir / "all_detailed.csv", detailed)
    atomic_csv(run_dir / "paper_protocol_detailed.csv", paper_detailed)
    atomic_csv(run_dir / "paper_protocol_summary.csv", paper_summary)
    atomic_csv(run_dir / "matched_total_budget_detailed.csv", budget_detailed)
    atomic_csv(run_dir / "matched_total_budget_summary.csv", budget_summary)
    paper_view_paths = export_paper_views(detailed, run_dir / "paper_views")
    (run_dir / "table_paper_protocol.tex").write_text(
        summary_to_latex(
            paper_summary,
            caption="Finite-atom convex-hull IOD LP relaxation and linear-surrogate baselines under the paper protocol.",
            label="tab:iod-cpu-v3-paper",
        ),
        encoding="utf-8",
    )
    (run_dir / "table_matched_total_budget.tex").write_text(
        summary_to_latex(
            budget_summary,
            caption="Matched prompt-count comparison; IOD initialization is included, but respondent effort is not equated.",
            label="tab:iod-cpu-v3-budget",
        ),
        encoding="utf-8",
    )

    manifest = {
        "status": "completed",
        "release_build": RELEASE_BUILD,
        "implementation": (
            "CPU finite-atom convex-hull IOD LP relaxation; every comparison row uses "
            "the formal safe outer output"
        ),
        "cache_schema": CACHE_SCHEMA,
        "power_sampling_policy": POWER_SAMPLING_POLICY,
        "iod_preprune_policy": IOD_PREPRUNE_POLICY,
        "concave_case_revision": CONCAVE_CASE_REVISION,
        "config_hash": config_hash,
        "config": asdict(config),
        "workbook_sha256": workbook_digest,
        "workbook_shape": [414, 4],
        "paper_alignment": (
            "paper mode uses a finite literal x**m atom grid with m>0 and a convex-hull LP relaxation; "
            "it is not the continuous one-exponent-per-coordinate model; signed_monotone is an explicitly "
            "labeled shifted Box-Cox extension"
        ),
        "initialization_scope": (
            "Under the primary elicited_band protocol, the simulated exponent-range answer reveals the "
            "selected close band but not the coordinate-specific powers. The full_dictionary_stress "
            "ablation deliberately withholds the close band."
        ),
        "question_count_scope": (
            "matched prompt counts do not establish equal respondent effort across prompt formats"
        ),
        "baseline_group_size_policy": "Squeeze-u, MinR, and MinD use s=d by default",
        "iod_runtime_scope": (
            "feature construction, simulated initialization, adaptive interaction, and LP-relaxation classification"
        ),
        "lp_numerical_policy": (
            "HiGHS status 4 triggers scaled dual-simplex and interior-point retries, then a phase-I "
            "feasibility solve for possible membership; any remaining ambiguity is retained in the safe "
            "outer, with ambiguity counts exported"
        ),
        "symmetric_difference_error_definition": "(FP+FN)/|I|",
        "formal_alpha_spill_definition": (
            "max(0, max_{p in S\\I} f(p*)-(1+epsilon)f(p)); it is a formal "
            "alpha-approximation certificate only when FN=0"
        ),
        "experiment_views": {
            name: str(path.relative_to(run_dir))
            for name, path in paper_view_paths.items()
        },
        "scalability_scope": (
            "All configured n values are actually executed. Full finite-LP classification is allowed "
            "only through scalability_full_lp_max_n. Larger cases export certificate-safe lower/upper "
            "bounds, unresolved counts, and classification_complete=false unless the bounds close."
        ),
        "case_manifest": case_manifest,
        "completed_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    }
    atomic_json(run_dir / "run_manifest.json", manifest)

    archive = root / "IOD_Nonlinear_CPU_V3_1_CONCAVE_SAFE_Results.zip"
    temporary_archive = archive.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary_archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(run_dir.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=str(path.relative_to(run_dir)))
    temporary_archive.replace(archive)

    print("\nPaper-protocol summary", flush=True)
    print(paper_summary.to_string(index=False), flush=True)
    print("\nMatched-total-budget summary", flush=True)
    print(budget_summary.to_string(index=False), flush=True)
    print(f"\nResults: {run_dir}\nZIP: {archive}", flush=True)
    return {
        "config_hash": config_hash,
        "run_dir": run_dir,
        "archive": archive,
        "detailed": detailed,
        "summary": summary,
        "scalability": scalability_result,
        "manifest": manifest,
    }


def main() -> int:
    run_experiment_suite(make_config())
    return 0

# ---------------- Inlined V3 scalability helpers ----------------

import hashlib
import inspect
import math
import os
import time
import warnings
from dataclasses import dataclass, field
from typing import Any, Callable, Iterable, Literal, Mapping, Sequence

for _thread_variable in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(_thread_variable, "1")
os.environ.setdefault("HIGHS_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.optimize import OptimizeWarning, linprog


Array = np.ndarray
DEFAULT_TOLERANCE = 1e-8

TOP20_METHOD = "Top-20 linear oracle weights"
SKYLINE_METHOD = "Skyline"
SQUEEZE_METHOD = "Squeeze-u linear surrogate"
MINR_METHOD = "MinR sampled linear surrogate"
MIND_METHOD = "MinD sampled linear surrogate"
IOD_CORE_METHOD = "IOD (LP relaxation) robust core"
IOD_OUTER_METHOD = "IOD (LP relaxation) safe outer"
SCALABILITY_METHOD_ORDER = (
    TOP20_METHOD,
    SKYLINE_METHOD,
    SQUEEZE_METHOD,
    MINR_METHOD,
    MIND_METHOD,
    IOD_CORE_METHOD,
    IOD_OUTER_METHOD,
)


def _matrix(values: Any, *, name: str = "points") -> Array:
    matrix = np.asarray(values, dtype=np.float64)
    if matrix.ndim != 2 or matrix.shape[0] == 0 or matrix.shape[1] == 0:
        raise ValueError(f"{name} must be a nonempty two-dimensional matrix")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{name} contains nonfinite values")
    return np.ascontiguousarray(matrix)


@dataclass(slots=True)
class _OrthantNode:
    lower: Array
    upper: Array
    indices: Array | None
    left: "_OrthantNode | None" = None
    right: "_OrthantNode | None" = None

    @property
    def leaf(self) -> bool:
        return self.indices is not None


class OrthantDominanceTree:
    """Static exact orthant range-emptiness tree.

    ``find_dominator(t)`` finds an input row ``x`` satisfying ``x >= t`` in
    every coordinate.  With ``require_strict=True`` it additionally requires
    ``x > t`` in at least one coordinate.  Bounding boxes only prune impossible
    branches; all accepted witnesses are checked against the original rows.
    Thus the result is identical to a brute-force comparison, including zeros,
    ties, duplicates, and an excluded self index.
    """

    def __init__(self, points: Any, *, leaf_size: int = 32) -> None:
        self.points = _matrix(points)
        self.n, self.d = self.points.shape
        self.leaf_size = int(leaf_size)
        if self.leaf_size < 1:
            raise ValueError("leaf_size must be positive")
        indices = np.arange(self.n, dtype=np.int64)
        self._root = self._build(indices)

    def _build(self, indices: Array) -> _OrthantNode:
        block = self.points[indices]
        lower = np.min(block, axis=0)
        upper = np.max(block, axis=0)
        if len(indices) <= self.leaf_size or not np.any(upper > lower):
            return _OrthantNode(lower, upper, indices.copy())
        spread = upper - lower
        dimension = int(np.argmax(spread))
        # Stable sorting gives deterministic trees and avoids an empty child
        # when many values equal the median.
        order = np.argsort(block[:, dimension], kind="stable")
        middle = len(indices) // 2
        left_indices = indices[order[:middle]]
        right_indices = indices[order[middle:]]
        if len(left_indices) == 0 or len(right_indices) == 0:
            return _OrthantNode(lower, upper, indices.copy())
        return _OrthantNode(
            lower,
            upper,
            None,
            self._build(left_indices),
            self._build(right_indices),
        )

    def find_dominator(
        self,
        threshold: Sequence[float],
        *,
        exclude_index: int | None = None,
        require_strict: bool = False,
    ) -> int | None:
        """Return an exact witness index, or ``None`` if the orthant is empty."""

        target = np.asarray(threshold, dtype=np.float64)
        if target.shape != (self.d,) or not np.isfinite(target).all():
            raise ValueError(f"threshold must be a finite vector of length {self.d}")
        excluded = None if exclude_index is None else int(exclude_index)
        if excluded is not None and not 0 <= excluded < self.n:
            raise IndexError("exclude_index is outside the input")

        stack = [self._root]
        while stack:
            node = stack.pop()
            if np.any(node.upper < target):
                continue
            # If no coordinate maximum is strictly above the threshold, no row
            # in this subtree can satisfy the strict part of the query.
            if require_strict and not np.any(node.upper > target):
                continue
            if node.leaf:
                assert node.indices is not None
                rows = self.points[node.indices]
                keep = np.all(rows >= target, axis=1)
                if require_strict:
                    keep &= np.any(rows > target, axis=1)
                if excluded is not None:
                    keep &= node.indices != excluded
                positions = np.flatnonzero(keep)
                if len(positions):
                    return int(node.indices[int(positions[0])])
                continue
            assert node.left is not None and node.right is not None
            # Search the child with the larger worst-coordinate surplus first;
            # this changes speed only, never the answer.
            children = (node.left, node.right)
            priority = [float(np.min(child.upper - target)) for child in children]
            first = int(priority[1] > priority[0])
            stack.append(children[1 - first])
            stack.append(children[first])
        return None

    def any_dominator(
        self,
        threshold: Sequence[float],
        *,
        exclude_index: int | None = None,
        require_strict: bool = False,
    ) -> bool:
        return self.find_dominator(
            threshold,
            exclude_index=exclude_index,
            require_strict=require_strict,
        ) is not None

    def dominated_mask(
        self,
        thresholds: Any | None = None,
        *,
        factor: float = 1.0,
        exclude_self: bool = True,
        require_strict: bool = True,
    ) -> Array:
        """Return one exact orthant-emptiness result per threshold row."""

        if not np.isfinite(factor) or factor < 0:
            raise ValueError("factor must be finite and nonnegative")
        queries = self.points if thresholds is None else _matrix(thresholds, name="thresholds")
        if queries.shape[1] != self.d:
            raise ValueError("threshold width differs from the tree width")
        if exclude_self and len(queries) != self.n:
            raise ValueError("exclude_self requires one threshold per input row")
        output = np.zeros(len(queries), dtype=bool)
        for index, row in enumerate(queries):
            output[index] = self.any_dominator(
                float(factor) * row,
                exclude_index=index if exclude_self else None,
                require_strict=require_strict,
            )
        return output


def skyline_indices(X: Any, *, leaf_size: int = 32) -> Array:
    """Exact Pareto skyline with the experiment's strict-dominance semantics."""

    values = _matrix(X, name="X")
    tree = OrthantDominanceTree(values, leaf_size=leaf_size)
    dominated = tree.dominated_mask(
        factor=1.0, exclude_self=True, require_strict=True
    )
    return np.flatnonzero(~dominated)


def epsilon_front_indices(
    X: Any, epsilon: float, *, leaf_size: int = 32
) -> Array:
    """Exact ``(1+epsilon)``-front without the quadratic pair matrix."""

    values = _matrix(X, name="X")
    epsilon = float(epsilon)
    if not np.isfinite(epsilon) or epsilon < 0:
        raise ValueError("epsilon must be finite and nonnegative")
    tree = OrthantDominanceTree(values, leaf_size=leaf_size)
    dominated = tree.dominated_mask(
        factor=1.0 + epsilon,
        exclude_self=True,
        require_strict=True,
    )
    return np.flatnonzero(~dominated)


def iod_epsilon_front_indices(
    X: Any,
    epsilon: float,
    *,
    utility_mode: Literal["paper", "signed_monotone"],
    allowed_powers: Sequence[float],
    leaf_size: int = 32,
) -> tuple[Array, dict[str, Any]]:
    """Scalable counterpart of the main IOD power-safe front.

    Linear baselines continue to call :func:`epsilon_front_indices` with the
    unmodified epsilon.  IOD uses the elicited monomial band: if its minimum
    exponent is below one, the raw factor is raised to ``1/m_min``.  The
    nonhomogeneous signed extension safely disables raw Stage-0 pruning.
    """

    values = _matrix(X, name="X")
    epsilon = float(epsilon)
    if not np.isfinite(epsilon) or epsilon < 0.0:
        raise ValueError("epsilon must be finite and nonnegative")
    powers = np.asarray(allowed_powers, dtype=np.float64)
    if powers.ndim != 1 or len(powers) == 0 or not np.isfinite(powers).all():
        raise ValueError("allowed_powers must be a nonempty finite vector")
    minimum_power = float(np.min(powers))
    if utility_mode == "signed_monotone":
        return np.arange(len(values), dtype=np.int64), {
            "status": "disabled_nonhomogeneous_signed_basis",
            "utility_epsilon": epsilon,
            "raw_epsilon": 0.0,
            "raw_factor": 1.0,
            "minimum_allowed_power": minimum_power,
        }
    if utility_mode != "paper":
        raise ValueError(utility_mode)
    if np.any(powers <= 0.0):
        raise ValueError("paper-mode allowed powers must be strictly positive")
    utility_factor = 1.0 + epsilon
    if minimum_power < 1.0:
        raw_factor = float(np.exp(np.log1p(epsilon) / minimum_power))
        status = "concave_adjusted"
    else:
        raw_factor = utility_factor
        status = "paper_raw_factor_valid"
    raw_epsilon = raw_factor - 1.0
    return epsilon_front_indices(
        values, raw_epsilon, leaf_size=leaf_size
    ), {
        "status": status,
        "utility_epsilon": epsilon,
        "raw_epsilon": raw_epsilon,
        "raw_factor": raw_factor,
        "minimum_allowed_power": minimum_power,
    }


@dataclass(frozen=True)
class _LPResult:
    status: Literal["optimal", "infeasible", "ambiguous"]
    value: float | None = None
    x: Array | None = None
    message: str = ""


def _polytope_arrays(polytope: Any, width: int) -> tuple[Array | None, Array | None]:
    if not hasattr(polytope, "arrays"):
        raise TypeError("interaction.polytope must provide arrays()")
    A, b = polytope.arrays()
    if A is None:
        return None, None
    matrix = np.asarray(A, dtype=np.float64)
    bounds = np.asarray(b, dtype=np.float64)
    if matrix.ndim != 2 or matrix.shape[1] != width or bounds.shape != (len(matrix),):
        raise ValueError("polytope arrays have incompatible shapes")
    if not np.isfinite(matrix).all() or not np.isfinite(bounds).all():
        raise ValueError("polytope arrays contain nonfinite values")
    return matrix, bounds


def _solve_simplex_lp(
    objective: Array,
    A: Array | None,
    b: Array | None,
    *,
    maximize: bool = False,
) -> _LPResult:
    """Solve a simplex LP, retrying uncertain/infeasible statuses safely."""

    direction = np.asarray(objective, dtype=np.float64)
    width = len(direction)
    c = -direction if maximize else direction
    attempts: list[tuple[str, int, str]] = []
    infeasible_methods: set[str] = set()
    for method, presolve, scaled in (
        ("highs-ds", True, False),
        ("highs-ipm", False, True),
        ("highs-ds", False, True),
    ):
        solve_A, solve_b = A, b
        if scaled and A is not None and b is not None and len(A):
            scales = np.maximum(np.max(np.abs(A), axis=1), np.abs(b))
            scales = np.where(scales > 1e-12, scales, 1.0)
            solve_A = A / scales[:, None]
            solve_b = b / scales
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=OptimizeWarning)
            result = linprog(
                c,
                A_ub=solve_A,
                b_ub=solve_b,
                A_eq=np.ones((1, width), dtype=np.float64),
                b_eq=np.ones(1, dtype=np.float64),
                bounds=[(0.0, None)] * width,
                method=method,
                options={
                    "presolve": bool(presolve),
                    "primal_feasibility_tolerance": 1e-9 if not scaled else 1e-8,
                    "dual_feasibility_tolerance": 1e-9 if not scaled else 1e-8,
                    "ipm_optimality_tolerance": 1e-9,
                },
            )
        attempts.append((method, int(result.status), str(result.message)))
        if result.status == 2:
            infeasible_methods.add(method)
            continue
        if not result.success or result.x is None or result.fun is None:
            continue
        x = np.asarray(result.x, dtype=np.float64)
        violation = max(abs(float(x.sum()) - 1.0), max(0.0, -float(x.min())))
        if A is not None and b is not None and len(A):
            violation = max(violation, float(np.max(A @ x - b)))
        if violation > 5e-7:
            attempts.append((method, 4, f"unchecked primal residual {violation:g}"))
            continue
        value = -float(result.fun) if maximize else float(result.fun)
        return _LPResult("optimal", value, x, str(result.message))
    # Agreement by two genuinely different HiGHS algorithms is required
    # before infeasibility removes a point from an outer upper bound.
    if {"highs-ds", "highs-ipm"} <= infeasible_methods:
        return _LPResult("infeasible", message="two HiGHS algorithms reported infeasible")
    detail = " | ".join(f"{m} status {s}: {msg}" for m, s, msg in attempts)
    return _LPResult("ambiguous", message=detail)


def _score_box(Phi: Array, lower: Array, upper: Array) -> tuple[Array, Array]:
    positive = np.maximum(Phi, 0.0)
    negative = np.minimum(Phi, 0.0)
    return positive @ lower + negative @ upper, positive @ upper + negative @ lower


def _target_from_scores(scores: Array, factor: float, tolerance: float) -> Array:
    return np.flatnonzero(factor * scores >= float(np.max(scores)) - tolerance)


def _outer_status(
    Phi: Array,
    p: int,
    A: Array | None,
    b: Array | None,
    factor: float,
    anchor: Array,
    tolerance: float,
) -> tuple[Literal["possible", "impossible", "ambiguous"], Array | None]:
    anchor_scores = Phi @ anchor
    gaps = anchor_scores - factor * float(Phi[p] @ anchor)
    if float(np.max(gaps)) <= tolerance:
        return "possible", anchor.copy()
    active = [int(np.argmax(gaps))]
    active_set = set(active)
    for _ in range(len(Phi) + 1):
        rows = Phi[np.asarray(active, dtype=int)] - factor * Phi[p]
        local_A = rows if A is None else np.vstack((A, rows))
        zeros = np.zeros(len(rows), dtype=np.float64)
        local_b = zeros if b is None else np.concatenate((b, zeros))
        solved = _solve_simplex_lp(np.zeros(Phi.shape[1]), local_A, local_b)
        if solved.status == "infeasible":
            return "impossible", None
        if solved.status != "optimal" or solved.x is None:
            return "ambiguous", None
        witness = solved.x
        gaps = Phi @ witness - factor * float(Phi[p] @ witness)
        q = int(np.argmax(gaps))
        if float(gaps[q]) <= tolerance:
            return "possible", witness
        if q in active_set:
            return "ambiguous", None
        active.append(q)
        active_set.add(q)
    return "ambiguous", None


def _linear_box_upper(vector: Array, lower: Array, upper: Array) -> float:
    return float(np.maximum(vector, 0.0) @ upper + np.minimum(vector, 0.0) @ lower)


def _core_status(
    Phi: Array,
    p: int,
    A: Array | None,
    b: Array | None,
    factor: float,
    competitor_order: Array,
    coefficient_lower: Array,
    coefficient_upper: Array,
    tolerance: float,
) -> Literal["necessary", "not_necessary", "ambiguous"]:
    for q in competitor_order:
        delta = Phi[int(q)] - factor * Phi[p]
        # The independently valid coefficient box gives a cheap rigorous upper
        # bound; no LP is needed when even this relaxation cannot violate p.
        if _linear_box_upper(delta, coefficient_lower, coefficient_upper) <= tolerance:
            continue
        solved = _solve_simplex_lp(delta, A, b, maximize=True)
        if solved.status != "optimal" or solved.value is None:
            return "ambiguous"
        if solved.value > tolerance:
            return "not_necessary"
    return "necessary"


@dataclass(frozen=True)
class IODAnytimeBounds:
    """Certificate-safe set bounds for one finite-atom IOD interaction."""

    outer_lower: Array
    outer_upper: Array
    core_lower: Array
    core_upper: Array
    completion_status: str
    full_classification_allowed: bool
    component_times: Mapping[str, float]
    core_computed: bool = True
    outer_ambiguous_count: int = 0
    core_ambiguous_count: int = 0
    exact_outer_attempted: int = 0
    exact_core_attempted: int = 0
    diagnostics: Mapping[str, Any] = field(default_factory=dict)

    @property
    def outer_unresolved_count(self) -> int:
        return int(len(np.setdiff1d(self.outer_upper, self.outer_lower, assume_unique=True)))

    @property
    def core_unresolved_count(self) -> int:
        return int(len(np.setdiff1d(self.core_upper, self.core_lower, assume_unique=True)))

    @property
    def complete(self) -> bool:
        return self.outer_unresolved_count == 0 and (
            not self.core_computed or self.core_unresolved_count == 0
        )

    def summary(self) -> dict[str, Any]:
        return {
            "outer_size_lower": int(len(self.outer_lower)),
            "outer_size_upper": int(len(self.outer_upper)),
            "core_size_lower": int(len(self.core_lower)),
            "core_size_upper": int(len(self.core_upper)),
            "outer_unresolved_count": self.outer_unresolved_count,
            "core_unresolved_count": self.core_unresolved_count,
            "outer_ambiguous_count": int(self.outer_ambiguous_count),
            "core_ambiguous_count": int(self.core_ambiguous_count),
            "core_computed": bool(self.core_computed),
            "exact_outer_attempted": int(self.exact_outer_attempted),
            "exact_core_attempted": int(self.exact_core_attempted),
            "completion_status": self.completion_status,
            "classification_complete": self.complete,
            "full_classification_allowed": bool(self.full_classification_allowed),
            **{str(key): float(value) for key, value in self.component_times.items()},
        }


def compute_iod_anytime_bounds(
    interaction: Any,
    epsilon: float,
    *,
    full_classification_max_n: int = 1_000,
    large_n_exact_candidates_per_set: int = 0,
    time_limit_seconds: float | None = None,
    anchor_directions: int = 4,
    seed: int = 0,
    compute_core: bool = True,
    tolerance: float = DEFAULT_TOLERANCE,
) -> IODAnytimeBounds:
    """Bound or fully classify the finite-atom IOD core and outer set.

    For ``n <= full_classification_max_n`` every still-unresolved point is sent
    through the finite LP classification (unless a time limit intervenes).  At
    larger ``n`` only the requested number of candidates per set is refined;
    the remaining exact set is represented by certified lower/upper bounds.
    No result at 10^4 or 10^5 is mislabeled as full classification unless its
    bounds happen to close without enumerating every tuple.
    """

    total_started = time.perf_counter()
    times: dict[str, float] = {}
    if not hasattr(interaction, "Phi") or not hasattr(interaction, "polytope"):
        raise TypeError("interaction must provide Phi and polytope fields")
    Phi = _matrix(interaction.Phi, name="interaction.Phi")
    n, width = Phi.shape
    epsilon = float(epsilon)
    if not np.isfinite(epsilon) or epsilon < 0:
        raise ValueError("epsilon must be finite and nonnegative")
    if int(full_classification_max_n) < 0:
        raise ValueError("full_classification_max_n must be nonnegative")
    if int(large_n_exact_candidates_per_set) < 0:
        raise ValueError("large_n_exact_candidates_per_set must be nonnegative")
    if time_limit_seconds is not None and (
        not np.isfinite(time_limit_seconds) or time_limit_seconds <= 0
    ):
        raise ValueError("time_limit_seconds must be finite and positive")
    factor = 1.0 + epsilon
    A, b = _polytope_arrays(interaction.polytope, width)

    started = time.perf_counter()
    feasible = _solve_simplex_lp(np.zeros(width), A, b)
    if feasible.status != "optimal" or feasible.x is None:
        raise RuntimeError(f"the interaction polytope is not certifiably feasible: {feasible.message}")
    base_anchor = feasible.x
    times["feasible_anchor_seconds"] = time.perf_counter() - started

    # Independent min/max bounds for every coefficient.  If a bound solve is
    # ambiguous, [0,1] is retained, which is weaker but still safe.
    started = time.perf_counter()
    coefficient_lower = np.zeros(width, dtype=np.float64)
    coefficient_upper = np.ones(width, dtype=np.float64)
    bound_margin = 5e-7
    coefficient_bound_ambiguities = 0
    for coordinate in range(width):
        unit = np.zeros(width, dtype=np.float64)
        unit[coordinate] = 1.0
        low = _solve_simplex_lp(unit, A, b, maximize=False)
        high = _solve_simplex_lp(unit, A, b, maximize=True)
        if low.status == "optimal" and low.value is not None:
            coefficient_lower[coordinate] = max(
                0.0, float(low.value) - bound_margin
            )
        else:
            coefficient_bound_ambiguities += 1
        if high.status == "optimal" and high.value is not None:
            coefficient_upper[coordinate] = min(
                1.0, float(high.value) + bound_margin
            )
        else:
            coefficient_bound_ambiguities += 1
    times["coefficient_bounds_seconds"] = time.perf_counter() - started

    started = time.perf_counter()
    anchors = [base_anchor]
    rng = np.random.default_rng(int(seed))
    for _ in range(max(0, int(anchor_directions))):
        solved = _solve_simplex_lp(rng.standard_normal(width), A, b, maximize=True)
        if solved.status == "optimal" and solved.x is not None:
            anchors.append(solved.x)
    # Rounded keys deduplicate numerically identical anchors, but the original
    # feasible vectors—not their rounded, potentially infeasible versions—are
    # retained for every certificate.
    unique_anchors: list[Array] = []
    seen_anchor_keys: set[tuple[float, ...]] = set()
    for anchor in anchors:
        key = tuple(np.round(anchor, 12).tolist())
        if key not in seen_anchor_keys:
            seen_anchor_keys.add(key)
            unique_anchors.append(anchor)
    anchor_matrix = np.vstack(unique_anchors)
    anchor_scores = anchor_matrix @ Phi.T
    times["anchor_generation_seconds"] = time.perf_counter() - started

    started = time.perf_counter()
    score_lower, score_upper = _score_box(Phi, coefficient_lower, coefficient_upper)
    best_lower = float(np.max(score_lower))
    best_upper = float(np.max(score_upper))
    # A point observed near-optimal at a feasible anchor is definitely possible.
    outer_lower_set: set[int] = set()
    for scores in anchor_scores:
        outer_lower_set.update(map(int, _target_from_scores(scores, factor, tolerance)))
    # If factor*sup score is below a competitor's universal lower score, the
    # point is definitely impossible.
    outer_upper_set = set(
        map(int, np.flatnonzero(factor * score_upper >= best_lower - tolerance))
    )
    outer_upper_set.update(outer_lower_set)

    if compute_core:
        # A core point must pass at every feasible anchor. Conversely, interval
        # dominance over all utilities certifies membership in the core lower set.
        anchor_pass = np.ones(n, dtype=bool)
        for scores in anchor_scores:
            anchor_pass &= factor * scores >= float(np.max(scores)) - tolerance
        core_upper_set = set(map(int, np.flatnonzero(anchor_pass)))
        core_lower_set = set(
            map(int, np.flatnonzero(factor * score_lower >= best_upper - tolerance))
        )
        core_upper_set.update(core_lower_set)
    else:
        core_upper_set = set()
        core_lower_set = set()
    times["initial_bounds_seconds"] = time.perf_counter() - started

    full_allowed = n <= int(full_classification_max_n)
    per_set_limit: int | None = None if full_allowed else int(large_n_exact_candidates_per_set)
    deadline = (
        None
        if time_limit_seconds is None
        else total_started + float(time_limit_seconds)
    )
    hit_time_limit = False

    outer_pending = sorted(
        outer_upper_set - outer_lower_set,
        key=lambda p: (score_upper[p], -p),
    )
    if per_set_limit is not None:
        outer_pending = outer_pending[:per_set_limit]
    outer_ambiguous = 0
    outer_attempted = 0
    started = time.perf_counter()
    for p in outer_pending:
        if deadline is not None and time.perf_counter() >= deadline:
            hit_time_limit = True
            break
        status, _ = _outer_status(Phi, p, A, b, factor, base_anchor, tolerance)
        outer_attempted += 1
        if status == "possible":
            outer_lower_set.add(p)
        elif status == "impossible":
            outer_upper_set.discard(p)
        else:
            outer_ambiguous += 1
    times["exact_outer_seconds"] = time.perf_counter() - started

    core_ambiguous = 0
    core_attempted = 0
    started = time.perf_counter()
    if compute_core:
        # Strong observed competitors are tried first, normally rejecting a
        # non-core point after one LP while preserving the all-q definition.
        competitor_upper = np.max(anchor_scores, axis=0)
        competitor_order = np.argsort(-competitor_upper, kind="stable")
        core_pending = sorted(
            core_upper_set - core_lower_set,
            key=lambda p: (-score_lower[p], p),
        )
        if per_set_limit is not None:
            core_pending = core_pending[:per_set_limit]
        for p in core_pending:
            if deadline is not None and time.perf_counter() >= deadline:
                hit_time_limit = True
                break
            status = _core_status(
                Phi,
                p,
                A,
                b,
                factor,
                competitor_order,
                coefficient_lower,
                coefficient_upper,
                tolerance,
            )
            core_attempted += 1
            if status == "necessary":
                core_lower_set.add(p)
            elif status == "not_necessary":
                core_upper_set.discard(p)
            else:
                core_ambiguous += 1
    times["exact_core_seconds"] = time.perf_counter() - started

    outer_lower = np.asarray(sorted(outer_lower_set), dtype=np.int64)
    outer_upper = np.asarray(sorted(outer_upper_set), dtype=np.int64)
    core_lower = np.asarray(sorted(core_lower_set), dtype=np.int64)
    core_upper = np.asarray(sorted(core_upper_set), dtype=np.int64)
    if not set(map(int, outer_lower)) <= set(map(int, outer_upper)):
        raise AssertionError("outer bounds are inconsistent")
    if not set(map(int, core_lower)) <= set(map(int, core_upper)):
        raise AssertionError("core bounds are inconsistent")
    outer_unresolved = len(set(map(int, outer_upper)) - set(map(int, outer_lower)))
    core_unresolved = len(set(map(int, core_upper)) - set(map(int, core_lower)))
    if outer_unresolved == 0 and (not compute_core or core_unresolved == 0):
        completion_status = "complete"
    elif hit_time_limit:
        completion_status = "time_limit"
    elif outer_ambiguous or core_ambiguous or coefficient_bound_ambiguities:
        completion_status = "bounded_numerical_ambiguity"
    elif not full_allowed:
        completion_status = "bounded_n_limit"
    else:
        completion_status = "bounded"
    times["total_anytime_seconds"] = time.perf_counter() - total_started
    return IODAnytimeBounds(
        outer_lower=outer_lower,
        outer_upper=outer_upper,
        core_lower=core_lower,
        core_upper=core_upper,
        completion_status=completion_status,
        full_classification_allowed=full_allowed,
        component_times=times,
        core_computed=bool(compute_core),
        outer_ambiguous_count=outer_ambiguous,
        core_ambiguous_count=core_ambiguous,
        exact_outer_attempted=outer_attempted,
        exact_core_attempted=core_attempted,
        diagnostics={
            "n": n,
            "coefficient_count": width,
            "anchor_count": len(anchor_matrix),
            "coefficient_bound_ambiguities": coefficient_bound_ambiguities,
            "core_computed": bool(compute_core),
            "fallback_policy": (
                "unresolved possible tuples remain in outer_upper"
                + (
                    "; unresolved universal tuples remain outside core_lower"
                    if compute_core
                    else ""
                )
            ),
        },
    )


def top20_linear(X: Any, weights: Sequence[float], k: int = 20) -> Array:
    values = _matrix(X, name="X")
    weights_array = np.asarray(weights, dtype=np.float64)
    if weights_array.shape != (values.shape[1],):
        raise ValueError("weights have the wrong width")
    count = min(max(0, int(k)), len(values))
    return np.argsort(-(values @ weights_array), kind="stable")[:count]


def squeeze_u_linear_scalable(
    X: Any,
    oracle: Any,
    *,
    epsilon: float,
    question_budget: int,
    group_size: int | None = None,
    leaf_size: int = 32,
) -> tuple[Array, int]:
    """Squeeze-u surrogate with the exact scalable epsilon front."""

    values = _matrix(X, name="X")
    candidate_indices = epsilon_front_indices(values, epsilon, leaf_size=leaf_size)
    D = values[candidate_indices]
    minimum, maximum = D.min(axis=0), D.max(axis=0)
    d = values.shape[1]
    group_size = d if group_size is None else int(group_size)
    if group_size < 2:
        raise ValueError("group_size must be at least two")
    artificial = np.repeat(minimum[None, :], d, axis=0)
    artificial[np.arange(d), np.arange(d)] = minimum + 0.5 * (maximum - minimum)
    questions = 0
    reference = 0
    i = 1
    while i < d and questions < int(question_budget):
        dimensions = [reference] + list(range(i, min(d, i + group_size - 1)))
        reference = int(dimensions[int(oracle.choose(artificial[dimensions]))])
        i += group_size - 1
        questions += 1
    lower, upper = np.zeros(d), np.ones(d)
    lower[reference] = upper[reference] = 1.0
    current = 0
    while questions < int(question_budget):
        if current == reference:
            current = (current + 1) % d
            continue
        grid = lower[current] + np.arange(group_size + 1) * (
            upper[current] - lower[current]
        ) / float(group_size)
        queries = np.zeros((group_size, d), dtype=np.float64)
        for k in range(1, group_size + 1):
            if k <= group_size - 1:
                queries[k - 1, reference] = grid[k:group_size].sum() / float(group_size)
            queries[k - 1, current] = k / float(group_size)
        chosen = int(oracle.choose(queries)) + 1
        lower[current], upper[current] = grid[chosen - 1], grid[chosen]
        questions += 1
        current = (current + 1) % d
    best_lower = float(np.max(D @ lower))
    keep = (1.0 + float(epsilon)) * (D @ upper) >= best_lower
    return candidate_indices[np.flatnonzero(keep)], questions


def _sample_feasible_weights(
    d: int,
    constraints: Sequence[Array],
    count: int,
    *,
    rng: np.random.Generator,
    max_tries: int | None = None,
) -> Array:
    limit = int(max_tries if max_tries is not None else 10 * int(count))
    matrix = np.vstack(constraints) if constraints else None
    accepted: list[Array] = []
    accepted_count = 0
    tries = 0
    while accepted_count < int(count) and tries < limit:
        batch = min(64, limit - tries)
        samples = rng.dirichlet(np.ones(d), size=batch)
        feasible = (
            np.ones(batch, dtype=bool)
            if matrix is None
            else np.all(matrix @ samples.T >= 0.0, axis=0)
        )
        if np.any(feasible):
            block = samples[feasible]
            accepted.append(block)
            accepted_count += len(block)
        tries += batch
    if not accepted:
        return np.empty((0, d), dtype=np.float64)
    return np.vstack(accepted)[: int(count)]


def minr_mind_linear_scalable(
    X: Any,
    oracle: Any,
    *,
    epsilon: float,
    algorithm: Literal["MinR", "MinD"],
    question_budget: int,
    seed: int,
    group_size: int | None = None,
    candidate_groups: int = 10,
    weight_samples: int = 100,
    leaf_size: int = 32,
) -> tuple[Array, int]:
    """MinR/MinD surrogate with the exact scalable epsilon front."""

    if algorithm not in {"MinR", "MinD"}:
        raise ValueError("algorithm must be MinR or MinD")
    values = _matrix(X, name="X")
    group_size = values.shape[1] if group_size is None else int(group_size)
    if group_size < 2:
        raise ValueError("group_size must be at least two")
    candidates = epsilon_front_indices(values, epsilon, leaf_size=leaf_size)
    constraints: list[Array] = []
    rng = np.random.default_rng(int(seed))
    questions = 0
    while questions < int(question_budget) and len(candidates) >= group_size:
        best_group: Array | None = None
        best_metric: float | None = None
        for _ in range(int(candidate_groups)):
            group = (
                candidates.copy()
                if len(candidates) == group_size
                else rng.choice(candidates, size=group_size, replace=False)
            )
            outcomes: list[float] = []
            for winner_position, winner_index in enumerate(group):
                temporary = list(constraints)
                winner = values[int(winner_index)]
                for other_position, other_index in enumerate(group):
                    if other_position != winner_position:
                        temporary.append(winner - values[int(other_index)])
                samples = _sample_feasible_weights(
                    values.shape[1], temporary, int(weight_samples), rng=rng
                )
                if len(samples) == 0:
                    metric = math.inf
                elif algorithm == "MinR":
                    metric = float(np.max(samples.max(axis=0) - samples.min(axis=0)))
                else:
                    differences = samples[:, None, :] - samples[None, :, :]
                    metric = float(
                        np.sqrt(np.max(np.sum(differences * differences, axis=2)))
                    )
                outcomes.append(metric)
            group_metric = float(np.mean(outcomes))
            if best_metric is None or group_metric < best_metric:
                best_metric, best_group = group_metric, np.asarray(group, dtype=int)
        if best_group is None:
            break
        selected = int(best_group[int(oracle.choose(values[best_group]))])
        for other in best_group:
            if int(other) != selected:
                constraints.append(values[selected] - values[int(other)])
        questions += 1
        samples = _sample_feasible_weights(
            values.shape[1], constraints, int(weight_samples), rng=rng
        )
        if len(samples):
            lower, upper = samples.min(axis=0), samples.max(axis=0)
            D = values[candidates]
            best_lower = float(np.max(D @ lower))
            candidates = candidates[
                (1.0 + float(epsilon)) * (D @ upper) >= best_lower
            ]
    return np.asarray(candidates, dtype=np.int64), questions


@dataclass(frozen=True)
class ExperimentBindings:
    """Explicit adapter for experiment classes/functions used by the runner."""

    generate_anti_correlated: Callable[..., Array]
    make_true_utility: Callable[..., Any]
    nonlinear_oracle_type: Callable[..., Any]
    run_iod_interaction: Callable[..., Any]
    compute_relaxation_sets: Callable[..., Any]
    target_indices: Callable[[Sequence[float], float], Array]
    set_metrics: Callable[[Sequence[int], Sequence[int]], Mapping[str, Any]]

    @classmethod
    def from_module(cls, module: Any) -> "ExperimentBindings":
        required = (
            "generate_anti_correlated",
            "make_true_utility",
            "NonlinearOracle",
            "run_iod_interaction",
            "compute_relaxation_sets",
            "target_indices",
            "set_metrics",
        )
        missing = [name for name in required if not hasattr(module, name)]
        if missing:
            raise TypeError(f"experiment module is missing: {', '.join(missing)}")
        return cls(
            module.generate_anti_correlated,
            module.make_true_utility,
            module.NonlinearOracle,
            module.run_iod_interaction,
            module.compute_relaxation_sets,
            module.target_indices,
            module.set_metrics,
        )


@dataclass
class ScalabilitySuiteResult:
    rows: pd.DataFrame
    memberships: dict[str, Any]
    iod_bounds: dict[int, IODAnytimeBounds]


def _canonical_method_selection(methods: Iterable[str] | None) -> set[str]:
    if methods is None:
        return set(SCALABILITY_METHOD_ORDER)
    aliases = {
        "top20": TOP20_METHOD,
        "skyline": SKYLINE_METHOD,
        "squeeze-u": SQUEEZE_METHOD,
        "squeezeu": SQUEEZE_METHOD,
        "minr": MINR_METHOD,
        "mind": MIND_METHOD,
        "iod-core": IOD_CORE_METHOD,
        "iod-outer": IOD_OUTER_METHOD,
        "iod": IOD_OUTER_METHOD,
    }
    selected: set[str] = set()
    for method in methods:
        key = str(method)
        canonical = aliases.get(key.casefold(), key)
        if canonical == IOD_OUTER_METHOD and key.casefold() == "iod":
            selected.update((IOD_CORE_METHOD, IOD_OUTER_METHOD))
        elif canonical not in SCALABILITY_METHOD_ORDER:
            raise ValueError(f"unknown scalability method: {method}")
        else:
            selected.add(canonical)
    return selected


def _runner_row(
    *,
    method: str,
    n: int,
    d: int,
    epsilon: float,
    seed: int,
    runtime_seconds: float,
    output: Array,
    target: Array,
    bindings: ExperimentBindings,
    questions: int = 0,
    mandatory_questions: int = 0,
    size_lower: int | None = None,
    size_upper: int | None = None,
    classification_status: str = "complete",
    classification_complete: bool = True,
    full_classification_allowed: bool = True,
    extra: Mapping[str, Any] | None = None,
) -> dict[str, Any]:
    metrics = dict(bindings.set_metrics(output, target))
    row = {
        "protocol": "scalability_v3",
        "method": method,
        "n": int(n),
        "d": int(d),
        "epsilon": float(epsilon),
        "seed": int(seed),
        "runtime_seconds": float(runtime_seconds),
        "pairwise_questions": int(questions),
        "mandatory_questions": int(mandatory_questions),
        "total_questions": int(questions + mandatory_questions),
        "output_size_lower": int(len(output) if size_lower is None else size_lower),
        "output_size_upper": int(len(output) if size_upper is None else size_upper),
        "classification_status": classification_status,
        "classification_complete": bool(classification_complete),
        "full_classification_allowed": bool(full_classification_allowed),
        **metrics,
    }
    if extra:
        row.update(extra)
    return row


def run_scalability_suite(
    config: Any,
    bindings: ExperimentBindings,
    *,
    sizes: Sequence[int] | None = None,
    seed: int | None = None,
    dimension: int | None = None,
    epsilon: float | None = None,
    full_classification_max_n: int | None = None,
    large_n_exact_candidates_per_set: int = 0,
    iod_time_limit_seconds: float | None = None,
    anchor_directions: int | None = None,
    methods: Iterable[str] | None = None,
    data_factory: Callable[..., Array] | None = None,
    include_memberships: bool = True,
    leaf_size: int = 32,
) -> ScalabilitySuiteResult:
    """Run independent end-to-end scalability cases for all paper methods.

    Baselines do not share a timed epsilon-front cache: each runtime includes
    its own exact front construction.  IOD's two rows share one interaction and
    one bound/classification run because core and outer are joint outputs.
    """

    selected = _canonical_method_selection(methods)
    sizes = tuple(int(n) for n in (sizes or getattr(config, "scalability_sizes")))
    if not sizes or any(n < 1 for n in sizes):
        raise ValueError("sizes must contain positive integers")
    if seed is None:
        seed = int(tuple(getattr(config, "seeds"))[0])
    if dimension is None:
        dimension = int(
            getattr(
                config,
                "scalability_dimension",
                getattr(config, "matched_budget_dimension", 5),
            )
        )
    if epsilon is None:
        epsilon = float(
            getattr(
                config,
                "scalability_epsilon",
                getattr(config, "matched_budget_epsilon", 0.05),
            )
        )
    if full_classification_max_n is None:
        full_classification_max_n = int(
            getattr(config, "scalability_full_lp_max_n", 1_000)
        )
    if anchor_directions is None:
        anchor_directions = int(
            getattr(config, "scalability_anchor_directions", 4)
        )
    baseline_budget = int(getattr(config, "baseline_pairwise_questions", 10))
    iod_questions = int(getattr(config, "adaptive_pairwise_questions", 10))
    factory = data_factory or bindings.generate_anti_correlated
    rows: list[dict[str, Any]] = []
    memberships: dict[str, Any] = {}
    iod_bounds: dict[int, IODAnytimeBounds] = {}

    need_iod = bool({IOD_CORE_METHOD, IOD_OUTER_METHOD} & selected)
    need_core = IOD_CORE_METHOD in selected
    for n in sizes:
        X = _matrix(factory(int(n), int(dimension), seed=int(seed)), name="generated X")
        if X.shape != (int(n), int(dimension)):
            raise ValueError(f"data_factory returned {X.shape}, expected {(n, dimension)}")
        truth = bindings.make_true_utility(
            int(dimension),
            config.powers,
            seed=int(seed),
            max_power_gap=float(config.truth_power_max_gap),
        )
        oracle = bindings.nonlinear_oracle_type(
            config.powers, truth, config.utility_mode, config.signed_shift
        )
        true_scores = np.asarray(oracle.scores(X), dtype=np.float64)
        target = np.asarray(bindings.target_indices(true_scores, float(epsilon)), dtype=np.int64)
        selected_band = getattr(truth, "selected_power_band", None)
        case_memberships: dict[str, Any] = {
            "dataset_sha256": hashlib.sha256(
                np.ascontiguousarray(X, dtype=np.float64).tobytes()
            ).hexdigest(),
            "generated_data_seed": int(seed),
            "shape": [int(n), int(dimension)],
            "true_powers": np.asarray(truth.powers, dtype=np.float64).tolist(),
            "true_selected_power_band": (
                []
                if selected_band is None
                else np.asarray(selected_band, dtype=np.float64).tolist()
            ),
            "target": target.tolist(),
            "outputs": {},
        }

        if TOP20_METHOD in selected:
            started = time.perf_counter()
            output = top20_linear(X, truth.weights)
            runtime = time.perf_counter() - started
            rows.append(_runner_row(
                method=TOP20_METHOD, n=n, d=dimension, epsilon=epsilon, seed=seed,
                runtime_seconds=runtime, output=output, target=target, bindings=bindings,
            ))
            case_memberships["outputs"][TOP20_METHOD] = output.tolist()

        if SKYLINE_METHOD in selected:
            started = time.perf_counter()
            output = skyline_indices(X, leaf_size=leaf_size)
            runtime = time.perf_counter() - started
            rows.append(_runner_row(
                method=SKYLINE_METHOD, n=n, d=dimension, epsilon=epsilon, seed=seed,
                runtime_seconds=runtime, output=output, target=target, bindings=bindings,
            ))
            case_memberships["outputs"][SKYLINE_METHOD] = output.tolist()

        if SQUEEZE_METHOD in selected:
            started = time.perf_counter()
            output, questions = squeeze_u_linear_scalable(
                X, oracle, epsilon=epsilon, question_budget=baseline_budget,
                group_size=dimension, leaf_size=leaf_size,
            )
            runtime = time.perf_counter() - started
            rows.append(_runner_row(
                method=SQUEEZE_METHOD, n=n, d=dimension, epsilon=epsilon, seed=seed,
                runtime_seconds=runtime, output=output, target=target, bindings=bindings,
                questions=questions,
            ))
            case_memberships["outputs"][SQUEEZE_METHOD] = output.tolist()

        for method, algorithm, offset in (
            (MINR_METHOD, "MinR", 101),
            (MIND_METHOD, "MinD", 102),
        ):
            if method not in selected:
                continue
            started = time.perf_counter()
            output, questions = minr_mind_linear_scalable(
                X,
                oracle,
                epsilon=epsilon,
                algorithm=algorithm,  # type: ignore[arg-type]
                question_budget=baseline_budget,
                seed=int(seed) + offset,
                group_size=dimension,
                leaf_size=leaf_size,
            )
            runtime = time.perf_counter() - started
            rows.append(_runner_row(
                method=method, n=n, d=dimension, epsilon=epsilon, seed=seed,
                runtime_seconds=runtime, output=output, target=target, bindings=bindings,
                questions=questions,
            ))
            case_memberships["outputs"][method] = output.tolist()

        if need_iod:
            # Paper Stage 0 is part of IOD's own end-to-end time.  All LP
            # indices below are local to this exact (1+epsilon)-front and must
            # be mapped back before metrics, membership export, or handoff.
            allowed_iod_powers = (
                np.asarray(selected_band, dtype=np.float64)
                if getattr(config, "exponent_range_policy", "elicited_band")
                == "elicited_band"
                else np.asarray(config.powers, dtype=np.float64)
            )
            preprune_started = time.perf_counter()
            iod_candidate_indices, iod_preprune_diagnostics = (
                iod_epsilon_front_indices(
                    X,
                    float(epsilon),
                    utility_mode=config.utility_mode,
                    allowed_powers=allowed_iod_powers,
                    leaf_size=leaf_size,
                )
            )
            iod_preprune_seconds = time.perf_counter() - preprune_started
            missing_from_iod_front = np.setdiff1d(
                target, iod_candidate_indices, assume_unique=False
            )
            if len(missing_from_iod_front):
                raise AssertionError(
                    "scalable IOD power-safe front removed true targets: "
                    f"n={n}, epsilon={epsilon}, "
                    f"allowed_powers={allowed_iod_powers.tolist()}, "
                    f"raw_factor={iod_preprune_diagnostics['raw_factor']}, "
                    f"missing={missing_from_iod_front.tolist()}"
                )
            iod_X = X[iod_candidate_indices]
            interaction = bindings.run_iod_interaction(
                iod_X,
                truth,
                config,
                adaptive_questions=iod_questions,
                seed=int(seed),
            )
            if int(n) <= int(full_classification_max_n):
                exact_started = time.perf_counter()
                exact_kwargs: dict[str, Any] = {
                    "workers": int(getattr(config, "workers", 1))
                }
                if "compute_core" in inspect.signature(
                    bindings.compute_relaxation_sets
                ).parameters:
                    exact_kwargs["compute_core"] = bool(need_core)
                exact = bindings.compute_relaxation_sets(
                    interaction, float(epsilon), **exact_kwargs
                )
                outer_upper = np.asarray(exact.safe_outer, dtype=np.int64)
                outer_ambiguous_indices = np.asarray(
                    getattr(exact, "ambiguous_outer_indices", ()), dtype=np.int64
                )
                outer_lower = np.setdiff1d(
                    outer_upper, outer_ambiguous_indices, assume_unique=False
                )
                core_lower = (
                    np.asarray(exact.robust_core, dtype=np.int64)
                    if need_core
                    else np.empty(0, dtype=np.int64)
                )
                core_ambiguous_indices = np.asarray(
                    getattr(exact, "ambiguous_core_indices", ()), dtype=np.int64
                )
                core_upper = np.union1d(core_lower, core_ambiguous_indices)
                outer_ambiguous = int(
                    getattr(exact, "ambiguous_outer_count", 0)
                )
                core_ambiguous = (
                    int(getattr(exact, "ambiguous_core_count", 0))
                    if need_core
                    else 0
                )
                complete = outer_ambiguous == 0 and (
                    not need_core or core_ambiguous == 0
                )
                exact_seconds = time.perf_counter() - exact_started
                local_bounds = IODAnytimeBounds(
                    outer_lower=outer_lower,
                    outer_upper=outer_upper,
                    core_lower=core_lower,
                    core_upper=core_upper,
                    completion_status=(
                        "complete" if complete else "bounded_numerical_ambiguity"
                    ),
                    full_classification_allowed=True,
                    component_times={
                        "exact_full_classification_seconds": exact_seconds,
                        "total_anytime_seconds": exact_seconds,
                    },
                    core_computed=bool(need_core),
                    outer_ambiguous_count=outer_ambiguous,
                    core_ambiguous_count=core_ambiguous,
                    exact_outer_attempted=len(interaction.Phi),
                    exact_core_attempted=(len(interaction.Phi) if need_core else 0),
                    diagnostics={
                        "n": len(interaction.Phi),
                        "full_solver": "main finite-LP classifier with numerical retries",
                    },
                )
            else:
                local_bounds = compute_iod_anytime_bounds(
                    interaction,
                    float(epsilon),
                    full_classification_max_n=0,
                    large_n_exact_candidates_per_set=large_n_exact_candidates_per_set,
                    time_limit_seconds=iod_time_limit_seconds,
                    anchor_directions=anchor_directions,
                    seed=int(seed),
                    compute_core=bool(need_core),
                )
            def global_indices(local: Array) -> Array:
                return np.sort(iod_candidate_indices[np.asarray(local, dtype=int)])

            component_times = {
                "preprune_seconds": float(iod_preprune_seconds),
                **dict(local_bounds.component_times),
            }
            bounds = IODAnytimeBounds(
                outer_lower=global_indices(local_bounds.outer_lower),
                outer_upper=global_indices(local_bounds.outer_upper),
                core_lower=global_indices(local_bounds.core_lower),
                core_upper=global_indices(local_bounds.core_upper),
                completion_status=local_bounds.completion_status,
                full_classification_allowed=local_bounds.full_classification_allowed,
                component_times=component_times,
                core_computed=local_bounds.core_computed,
                outer_ambiguous_count=local_bounds.outer_ambiguous_count,
                core_ambiguous_count=local_bounds.core_ambiguous_count,
                exact_outer_attempted=local_bounds.exact_outer_attempted,
                exact_core_attempted=local_bounds.exact_core_attempted,
                diagnostics={
                    **dict(local_bounds.diagnostics),
                    "original_n": int(n),
                    "candidate_front_size": int(len(iod_candidate_indices)),
                    "preprune": iod_preprune_diagnostics,
                    "index_space": "global database row indices",
                },
            )
            iod_bounds[int(n)] = bounds
            joint_runtime = float(
                iod_preprune_seconds
                + getattr(interaction, "interaction_seconds", 0.0)
                + bounds.component_times["total_anytime_seconds"]
            )
            common_extra = {
                "candidate_front_size": int(len(iod_candidate_indices)),
                "outer_unresolved_count": bounds.outer_unresolved_count,
                "outer_ambiguous_count": bounds.outer_ambiguous_count,
                **{f"iod_{key}": value for key, value in bounds.component_times.items()},
            }
            if need_core:
                common_extra.update(
                    {
                        "core_unresolved_count": bounds.core_unresolved_count,
                        "core_ambiguous_count": bounds.core_ambiguous_count,
                    }
                )
            if IOD_CORE_METHOD in selected:
                # The certified lower set is always a valid robust-core output.
                output = bounds.core_lower
                rows.append(_runner_row(
                    method=IOD_CORE_METHOD, n=n, d=dimension, epsilon=epsilon, seed=seed,
                    runtime_seconds=joint_runtime, output=output, target=target,
                    bindings=bindings,
                    questions=int(getattr(interaction, "pairwise_questions", 0)),
                    mandatory_questions=int(getattr(interaction, "mandatory_questions", 0)),
                    size_lower=len(bounds.core_lower), size_upper=len(bounds.core_upper),
                    classification_status=bounds.completion_status,
                    classification_complete=bounds.complete,
                    full_classification_allowed=bounds.full_classification_allowed,
                    extra={**common_extra, "reported_bound": "core_lower"},
                ))
                case_memberships["outputs"][IOD_CORE_METHOD] = output.tolist()
            if IOD_OUTER_METHOD in selected:
                # The certified upper set is always safe to return as an outer.
                output = bounds.outer_upper
                rows.append(_runner_row(
                    method=IOD_OUTER_METHOD, n=n, d=dimension, epsilon=epsilon, seed=seed,
                    runtime_seconds=joint_runtime, output=output, target=target,
                    bindings=bindings,
                    questions=int(getattr(interaction, "pairwise_questions", 0)),
                    mandatory_questions=int(getattr(interaction, "mandatory_questions", 0)),
                    size_lower=len(bounds.outer_lower), size_upper=len(bounds.outer_upper),
                    classification_status=bounds.completion_status,
                    classification_complete=bounds.complete,
                    full_classification_allowed=bounds.full_classification_allowed,
                    extra={**common_extra, "reported_bound": "outer_upper"},
                ))
                case_memberships["outputs"][IOD_OUTER_METHOD] = output.tolist()
            case_memberships["iod_bounds"] = bounds.summary()
            case_memberships["iod_candidate_indices"] = iod_candidate_indices.tolist()
            case_memberships["iod_preprune_diagnostics"] = iod_preprune_diagnostics
        if include_memberships:
            memberships[str(n)] = case_memberships

    frame = pd.DataFrame(rows)
    if len(frame):
        order = {method: index for index, method in enumerate(SCALABILITY_METHOD_ORDER)}
        frame["_method_order"] = frame["method"].map(order)
        frame = frame.sort_values(["n", "_method_order"], kind="stable").drop(
            columns=["_method_order"]
        ).reset_index(drop=True)
    return ScalabilitySuiteResult(frame, memberships, iod_bounds)


__all__ = [
    "IODAnytimeBounds",
    "OrthantDominanceTree",
    "SCALABILITY_METHOD_ORDER",
    "ScalabilitySuiteResult",
    "ExperimentBindings",
    "compute_iod_anytime_bounds",
    "epsilon_front_indices",
    "iod_epsilon_front_indices",
    "minr_mind_linear_scalable",
    "run_scalability_suite",
    "skyline_indices",
    "squeeze_u_linear_scalable",
    "top20_linear",
]

# ----------------- Inlined V3 reporting helpers -----------------

import json
import math
import os
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd


REPORTING_SCHEMA = "iod-nonlinear-cpu-v3-reporting-20260722-r1"
IOD_SHARED_RUNTIME_NOTE = (
    "Comparison tables use IOD's safe outer output. If the joint robust-core "
    "audit output is present in imported data, its shared runtime must not be summed."
)
DELTA_NOTE = (
    "Every paired delta is IOD safe outer minus Squeeze-u. W/T/L uses the "
    "metric direction declared by reporting_v3 (lower or higher is better)."
)

METHOD_FAMILY_ORDER = ("IOD", "Squeeze-u", "Top-k", "Skyline", "MinR", "MinD")
METHOD_ORDER_INDEX = {name: position for position, name in enumerate(METHOD_FAMILY_ORDER)}

NUMERIC_COLUMNS = (
    "replicate",
    "seed",
    "n",
    "d",
    "epsilon",
    "total_budget_cap",
    "pairwise_questions",
    "mandatory_questions",
    "total_questions",
    "runtime_seconds",
    "output_size",
    "target_size",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "symmetric_difference_error",
    "alpha_spill",
    "alpha_spill_absolute",
    "alpha_spill_relative",
    "formal_alpha",
    "true_power_min",
    "true_power_max",
    "true_power_span",
    "true_power_max_allowed_gap",
    "configured_power_gap",
)

SUMMARY_METRICS = (
    "output_size",
    "target_size",
    "precision",
    "recall",
    "symmetric_difference_error",
    "alpha_spill",
    "formal_alpha",
    "runtime_seconds",
    "pairwise_questions",
    "mandatory_questions",
    "total_questions",
)

PAIR_METRIC_DIRECTIONS: Mapping[str, str] = {
    "output_size": "lower",
    "precision": "higher",
    "recall": "higher",
    "symmetric_difference_error": "lower",
    "alpha_spill": "lower",
    "formal_alpha": "lower",
    "runtime_seconds": "lower",
    "total_questions": "lower",
}

DEFAULT_SUMMARY_CONTEXT = (
    "protocol",
    "dataset",
    "n",
    "d",
    "epsilon",
    "total_budget_cap",
    "utility_mode",
    "exponent_range_policy",
    "power_regime",
)

DEFAULT_PAIR_KEYS = (
    "experiment_id",
    "case_id",
    "run_id",
    "protocol",
    "dataset",
    "replicate",
    "seed",
    "n",
    "d",
    "epsilon",
    "total_budget_cap",
    "utility_mode",
    "power_sampling_policy",
    "exponent_range_policy",
    "power_regime",
    "configured_power_gap",
)


@dataclass(frozen=True)
class AlphaSpillResult:
    """Formal bounded-spill result for one returned membership set.

    ``bounded_spill`` is always the maximum score violation among false
    positives (zero when there are none). ``formal_alpha`` is equal to that
    value only when ``certificate_valid`` is true; otherwise it is NaN because
    Definition 4 also requires zero false negatives.
    """

    formal_alpha: float
    bounded_spill: float
    certificate_valid: bool
    best_score: float
    target_threshold: float
    output_size: int
    target_size: int
    tp: int
    fp: int
    fn: int

    @property
    def alpha_spill(self) -> float:
        """Alias matching the canonical reporting-column name."""

        return self.bounded_spill

    @property
    def no_false_negatives(self) -> bool:
        return self.certificate_valid

    def as_dict(self, *, prefix: str = "") -> dict[str, Any]:
        values = asdict(self)
        values["alpha_spill"] = self.bounded_spill
        values["no_false_negatives"] = self.certificate_valid
        return {f"{prefix}{key}": value for key, value in values.items()}


def _membership_mask(membership: Any, n: int, *, name: str) -> np.ndarray:
    """Convert an index collection or boolean/0-1 mask into a boolean mask.

    Integer ``[0, 1]`` is inherently ambiguous when ``n == 2``.  We interpret
    it as an index collection because completed-case membership JSON stores
    indices; callers that mean a mask for such a tiny relation should pass a
    boolean array.
    """

    if membership is None:
        raise ValueError(f"{name} membership is required")
    if isinstance(membership, (set, frozenset)):
        membership = list(membership)
    array = np.asarray(membership)
    if array.ndim != 1:
        raise ValueError(f"{name} membership must be one-dimensional")
    if array.dtype == np.bool_:
        if len(array) != n:
            raise ValueError(f"{name} boolean mask has length {len(array)}, expected {n}")
        return array.astype(bool, copy=True)
    if len(array) == n and n > 2 and np.issubdtype(array.dtype, np.number):
        finite = np.isfinite(array.astype(float, copy=False))
        if finite.all() and np.isin(array, (0, 1)).all():
            return array.astype(bool)
    if len(array) == 0:
        return np.zeros(n, dtype=bool)
    if not np.issubdtype(array.dtype, np.integer):
        numeric = pd.to_numeric(pd.Series(array), errors="coerce").to_numpy(dtype=float)
        if not np.isfinite(numeric).all() or not np.equal(numeric, np.floor(numeric)).all():
            raise ValueError(f"{name} indices must be finite integers")
        indices = numeric.astype(np.int64)
    else:
        indices = array.astype(np.int64, copy=False)
    if np.any(indices < 0) or np.any(indices >= n):
        raise ValueError(f"{name} contains an index outside [0, {n})")
    mask = np.zeros(n, dtype=bool)
    mask[indices] = True
    return mask


def target_membership_from_scores(
    true_scores: Sequence[float], epsilon: float, *, atol: float = 1e-12
) -> np.ndarray:
    """Return the paper's multiplicative epsilon-indistinguishability target."""

    scores = np.asarray(true_scores, dtype=np.float64)
    if scores.ndim != 1 or len(scores) == 0 or not np.isfinite(scores).all():
        raise ValueError("true_scores must be a nonempty finite one-dimensional vector")
    if not np.isfinite(epsilon) or float(epsilon) < 0:
        raise ValueError("epsilon must be finite and nonnegative")
    scale = max(1.0, float(np.max(np.abs(scores))))
    if float(np.min(scores)) < -float(atol) * scale:
        raise ValueError("the multiplicative alpha-spill definition requires nonnegative scores")
    scores = np.maximum(scores, 0.0)
    best = float(np.max(scores))
    threshold = best / (1.0 + float(epsilon))
    return scores >= threshold - float(atol) * max(1.0, abs(best))


def formal_alpha_spill(
    true_scores: Sequence[float],
    epsilon: float,
    output_membership: Any,
    target_membership: Any | None = None,
    *,
    tp: int | None = None,
    fn: int | None = None,
    atol: float = 1e-12,
) -> AlphaSpillResult:
    """Compute bounded spill and formal alpha-certificate validity.

    Memberships may be index collections, boolean masks, or numeric 0/1 masks.
    When ``target_membership`` is omitted, it is derived exhaustively from the
    true scores using ``max(score)/(1+epsilon)``. Optional ``tp`` and ``fn``
    counts are consistency checks useful when enriching detailed result rows.
    """

    scores = np.asarray(true_scores, dtype=np.float64)
    target_from_scores = target_membership_from_scores(scores, epsilon, atol=atol)
    target = (
        target_from_scores
        if target_membership is None
        else _membership_mask(target_membership, len(scores), name="target")
    )
    output = _membership_mask(output_membership, len(scores), name="output")
    tp_actual = int(np.count_nonzero(output & target))
    fp_actual = int(np.count_nonzero(output & ~target))
    fn_actual = int(np.count_nonzero(~output & target))
    if tp is not None and int(tp) != tp_actual:
        raise ValueError(f"provided tp={tp} disagrees with memberships ({tp_actual})")
    if fn is not None and int(fn) != fn_actual:
        raise ValueError(f"provided fn={fn} disagrees with memberships ({fn_actual})")

    best = float(np.max(scores))
    threshold = best / (1.0 + float(epsilon))
    false_positive_scores = scores[output & ~target]
    if len(false_positive_scores):
        violations = best - (1.0 + float(epsilon)) * false_positive_scores
        bounded_spill = max(0.0, float(np.max(violations)))
    else:
        bounded_spill = 0.0
    valid = fn_actual == 0
    return AlphaSpillResult(
        formal_alpha=bounded_spill if valid else math.nan,
        bounded_spill=bounded_spill,
        certificate_valid=valid,
        best_score=best,
        target_threshold=threshold,
        output_size=int(np.count_nonzero(output)),
        target_size=int(np.count_nonzero(target)),
        tp=tp_actual,
        fp=fp_actual,
        fn=fn_actual,
    )


def _method_identity(label: Any) -> tuple[str, str, str]:
    text = str(label).strip()
    lowered = " ".join(text.lower().replace("_", "-").split())
    if "iod" in lowered:
        if any(token in lowered for token in ("core", "forall", "s-forall", "s∀")):
            return "IOD", "robust_core", "IOD (robust core)"
        if any(token in lowered for token in ("outer", "possible", "exists", "s-exists", "s∃")):
            return "IOD", "safe_outer", "IOD"
        return "IOD", "primary", "IOD"
    if "squeeze" in lowered:
        return "Squeeze-u", "primary", "Squeeze-u"
    if "skyline" in lowered:
        return "Skyline", "primary", "Skyline"
    if "minr" in lowered or "min-r" in lowered:
        return "MinR", "primary", "MinR"
    if "mind" in lowered or "min-d" in lowered:
        return "MinD", "primary", "MinD"
    if "top" in lowered and ("oracle" in lowered or "-k" in lowered or any(char.isdigit() for char in lowered)):
        return "Top-k", "primary", "Top-k"
    return text, "primary", text


def _parse_number_sequence(value: Any) -> np.ndarray:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.empty(0, dtype=float)
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return np.empty(0, dtype=float)
        try:
            value = json.loads(stripped)
        except json.JSONDecodeError:
            value = [part.strip() for part in stripped.strip("[](){}").split(",") if part.strip()]
    array = np.asarray(value, dtype=float).reshape(-1)
    return array[np.isfinite(array)]


def _safe_ratio(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator > 0 else math.nan


def _coerce_boolean(series: pd.Series) -> pd.Series:
    """Coerce common CSV boolean spellings while preserving missing values."""

    if pd.api.types.is_bool_dtype(series.dtype):
        return series.astype("boolean")
    lowered = series.astype("string").str.strip().str.lower()
    mapped = lowered.map(
        {
            "true": True,
            "1": True,
            "yes": True,
            "false": False,
            "0": False,
            "no": False,
        }
    )
    return mapped.astype("boolean")


def canonicalize_detailed_results(detailed: pd.DataFrame) -> pd.DataFrame:
    """Normalize one detailed result frame without mutating the caller's data."""

    if not isinstance(detailed, pd.DataFrame):
        raise TypeError("detailed must be a pandas DataFrame")
    if "method" not in detailed.columns:
        raise ValueError("canonical detailed results require a 'method' column")
    frame = detailed.copy(deep=True)
    frame["method_raw"] = frame["method"].astype(str)
    identities = frame["method_raw"].map(_method_identity)
    frame["method_family"] = identities.map(lambda value: value[0])
    frame["output_role"] = identities.map(lambda value: value[1])
    frame["method_display"] = identities.map(lambda value: value[2])
    frame["method_order"] = frame["method_family"].map(METHOD_ORDER_INDEX).fillna(999).astype(int)

    for column in NUMERIC_COLUMNS:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    if "total_questions" not in frame.columns and {
        "pairwise_questions",
        "mandatory_questions",
    } <= set(frame.columns):
        frame["total_questions"] = frame["pairwise_questions"].fillna(0) + frame[
            "mandatory_questions"
        ].fillna(0)

    if {"tp", "fp"} <= set(frame.columns):
        derived_precision = frame["tp"] / (frame["tp"] + frame["fp"]).replace(0, np.nan)
        frame["precision"] = (
            frame["precision"].where(frame["precision"].notna(), derived_precision)
            if "precision" in frame.columns
            else derived_precision
        )
        if "output_size" not in frame.columns:
            frame["output_size"] = frame["tp"] + frame["fp"]
    if {"tp", "fn"} <= set(frame.columns):
        derived_recall = frame["tp"] / (frame["tp"] + frame["fn"]).replace(0, np.nan)
        frame["recall"] = (
            frame["recall"].where(frame["recall"].notna(), derived_recall)
            if "recall" in frame.columns
            else derived_recall
        )
        if "target_size" not in frame.columns:
            frame["target_size"] = frame["tp"] + frame["fn"]
    if {"fp", "fn", "target_size"} <= set(frame.columns):
        derived_error = (frame["fp"] + frame["fn"]) / frame["target_size"].replace(0, np.nan)
        frame["symmetric_difference_error"] = (
            frame["symmetric_difference_error"].where(
                frame["symmetric_difference_error"].notna(), derived_error
            )
            if "symmetric_difference_error" in frame.columns
            else derived_error
        )
    if "alpha_spill" not in frame.columns:
        for candidate in ("alpha_spill_absolute", "bounded_spill"):
            if candidate in frame.columns:
                frame["alpha_spill"] = pd.to_numeric(frame[candidate], errors="coerce")
                break
    if "alpha_certificate_valid" not in frame.columns:
        if "alpha_approximation_valid" in frame.columns:
            frame["alpha_certificate_valid"] = _coerce_boolean(
                frame["alpha_approximation_valid"]
            )
        elif "fn" in frame.columns:
            frame["alpha_certificate_valid"] = frame["fn"].eq(0).astype("boolean")
    elif "alpha_certificate_valid" in frame.columns:
        frame["alpha_certificate_valid"] = _coerce_boolean(
            frame["alpha_certificate_valid"]
        )
    if "formal_alpha" not in frame.columns and "alpha_spill" in frame.columns:
        if "alpha_certificate_valid" in frame.columns:
            frame["formal_alpha"] = frame["alpha_spill"].where(
                frame["alpha_certificate_valid"].fillna(False), np.nan
            )
        elif "fn" in frame.columns:
            frame["formal_alpha"] = frame["alpha_spill"].where(frame["fn"].eq(0), np.nan)

    if "selected_power_band" in frame.columns:
        parsed = frame["selected_power_band"].map(_parse_number_sequence)
        frame["selected_power_band_min"] = parsed.map(
            lambda values: float(np.min(values)) if len(values) else math.nan
        )
        frame["selected_power_band_max"] = parsed.map(
            lambda values: float(np.max(values)) if len(values) else math.nan
        )
        frame["selected_power_band_width"] = (
            frame["selected_power_band_max"] - frame["selected_power_band_min"]
        )
        frame["selected_power_band_cardinality"] = parsed.map(len).astype(int)
    if "configured_power_gap" not in frame.columns:
        for candidate in (
            "true_power_max_allowed_gap",
            "power_band_max_gap",
            "max_power_gap",
            "power_band_width",
        ):
            if candidate in frame.columns:
                frame["configured_power_gap"] = pd.to_numeric(frame[candidate], errors="coerce")
                break
    if "configured_power_gap" not in frame.columns:
        frame["configured_power_gap"] = np.nan

    frame["runtime_accounting_note"] = np.where(
        frame["method_family"].eq("IOD"), IOD_SHARED_RUNTIME_NOTE, ""
    )
    return frame


def _present(frame: pd.DataFrame, columns: Iterable[str]) -> list[str]:
    return [column for column in columns if column in frame.columns]


def _finite_values(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return values[np.isfinite(values)]


def _describe(values: np.ndarray) -> dict[str, float | int | str]:
    if len(values) == 0:
        return {
            "n": 0,
            "mean": math.nan,
            "sd": math.nan,
            "median": math.nan,
            "q1": math.nan,
            "q3": math.nan,
            "iqr": math.nan,
            "mean_sd": "NA",
            "median_iqr": "NA",
        }
    mean = float(np.mean(values))
    sd = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
    median = float(np.median(values))
    q1, q3 = np.quantile(values, [0.25, 0.75])
    iqr = float(q3 - q1)
    return {
        "n": int(len(values)),
        "mean": mean,
        "sd": sd,
        "median": median,
        "q1": float(q1),
        "q3": float(q3),
        "iqr": iqr,
        "mean_sd": f"{mean:.4f} ± {sd:.4f}",
        "median_iqr": f"{median:.4f} [{float(q1):.4f}, {float(q3):.4f}]",
    }


def _add_descriptions(record: dict[str, Any], group: pd.DataFrame, metrics: Sequence[str]) -> None:
    for metric in metrics:
        if metric not in group.columns:
            continue
        for statistic, value in _describe(_finite_values(group[metric])).items():
            record[f"{metric}_{statistic}"] = value


def _group_records(
    frame: pd.DataFrame,
    group_columns: Sequence[str],
    *,
    metrics: Sequence[str] = SUMMARY_METRICS,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=list(group_columns) + ["runs"])
    records: list[dict[str, Any]] = []
    grouped: Iterable[tuple[Any, pd.DataFrame]]
    if group_columns:
        grouped = frame.groupby(list(group_columns), sort=False, dropna=False)
    else:
        grouped = [((), frame)]
    for keys, group in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)
        record = dict(zip(group_columns, keys))
        record["runs"] = int(len(group))
        _add_descriptions(record, group, metrics)

        if {"tp", "fp", "fn"} <= set(group.columns):
            tp = float(pd.to_numeric(group["tp"], errors="coerce").fillna(0).sum())
            fp = float(pd.to_numeric(group["fp"], errors="coerce").fillna(0).sum())
            fn = float(pd.to_numeric(group["fn"], errors="coerce").fillna(0).sum())
            record.update(
                {
                    "micro_tp": int(tp),
                    "micro_fp": int(fp),
                    "micro_fn": int(fn),
                    "micro_precision": _safe_ratio(tp, tp + fp),
                    "micro_recall": _safe_ratio(tp, tp + fn),
                    "micro_f1": _safe_ratio(2 * tp, 2 * tp + fp + fn),
                    "micro_symmetric_difference_error": _safe_ratio(fp + fn, tp + fn),
                    "zero_false_positive_rate": float(group["fp"].eq(0).mean()),
                    "zero_false_negative_rate": float(group["fn"].eq(0).mean()),
                }
            )
        for metric in ("precision", "recall", "symmetric_difference_error", "alpha_spill"):
            if metric in group.columns:
                values = _finite_values(group[metric])
                record[f"macro_{metric}"] = float(np.mean(values)) if len(values) else math.nan

        if "output_size" in group.columns:
            output = pd.to_numeric(group["output_size"], errors="coerce")
            nonempty = output.gt(0)
            record["nonempty_count"] = int(nonempty.sum())
            record["nonempty_coverage"] = float(nonempty.mean())
            is_core = bool(group["output_role"].eq("robust_core").all()) if "output_role" in group else False
            record["core_nonempty_count"] = int(nonempty.sum()) if is_core else math.nan
            record["core_nonempty_coverage"] = float(nonempty.mean()) if is_core else math.nan
            if "precision" in group.columns:
                precision_defined = group["precision"].notna()
                record["precision_defined_count"] = int(precision_defined.sum())
                record["precision_defined_coverage"] = float(precision_defined.mean())
        is_iod = bool(group["method_family"].eq("IOD").all()) if "method_family" in group else False
        record["runtime_accounting_note"] = IOD_SHARED_RUNTIME_NOTE if is_iod else ""
        records.append(record)
    return pd.DataFrame.from_records(records)


def summarize_results(
    detailed: pd.DataFrame,
    *,
    group_columns: Sequence[str] | None = None,
    metrics: Sequence[str] = SUMMARY_METRICS,
) -> pd.DataFrame:
    """Aggregate detailed rows with robust distribution and micro/macro metrics."""

    frame = canonicalize_detailed_results(detailed)
    if group_columns is None:
        group_columns = _present(
            frame,
            (*DEFAULT_SUMMARY_CONTEXT, "method_family", "method_display", "output_role"),
        )
    return _group_records(frame, list(group_columns), metrics=metrics)


def _sort_view(frame: pd.DataFrame) -> pd.DataFrame:
    sort_candidates = (
        "protocol",
        "dataset",
        "n",
        "d",
        "epsilon",
        "total_budget_cap",
        "configured_power_gap",
        "method_order",
        "output_role",
        "replicate",
        "seed",
    )
    columns = _present(frame, sort_candidates)
    return frame.sort_values(columns, kind="stable", na_position="last").reset_index(drop=True) if columns else frame.reset_index(drop=True)


def _infer_pair_keys(frame: pd.DataFrame) -> list[str]:
    keys = _present(frame, DEFAULT_PAIR_KEYS)
    if not keys:
        raise ValueError("could not infer paired case keys; pass pair_keys explicitly")
    return keys


def _outcome(iod: float, squeeze: float, direction: str, tolerance: float) -> str:
    if not (np.isfinite(iod) and np.isfinite(squeeze)):
        return "NA"
    scale = max(1.0, abs(iod), abs(squeeze))
    difference = iod - squeeze
    if abs(difference) <= tolerance * scale:
        return "T"
    iod_better = difference < 0 if direction == "lower" else difference > 0
    return "W" if iod_better else "L"


def _paired_summary(
    paired: pd.DataFrame,
    group_columns: Sequence[str],
    *,
    metric_directions: Mapping[str, str],
) -> pd.DataFrame:
    if paired.empty:
        return pd.DataFrame(columns=list(group_columns) + ["paired_runs"])
    records: list[dict[str, Any]] = []
    grouped: Iterable[tuple[Any, pd.DataFrame]] = (
        paired.groupby(list(group_columns), sort=False, dropna=False)
        if group_columns
        else [((), paired)]
    )
    for keys, group in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)
        record = dict(zip(group_columns, keys))
        record["paired_runs"] = int(len(group))
        for metric in metric_directions:
            iod_column = f"{metric}_iod"
            squeeze_column = f"{metric}_squeeze"
            delta_column = f"delta_{metric}"
            if iod_column not in group or squeeze_column not in group:
                continue
            for side, column in (("iod", iod_column), ("squeeze", squeeze_column), ("delta", delta_column)):
                if column not in group:
                    continue
                for statistic, value in _describe(_finite_values(group[column])).items():
                    record[f"{metric}_{side}_{statistic}"] = value
            outcome_column = f"outcome_{metric}"
            if outcome_column in group:
                counts = group[outcome_column].value_counts()
                comparable = int(sum(int(counts.get(value, 0)) for value in ("W", "T", "L")))
                record[f"{metric}_wins"] = int(counts.get("W", 0))
                record[f"{metric}_ties"] = int(counts.get("T", 0))
                record[f"{metric}_losses"] = int(counts.get("L", 0))
                record[f"{metric}_not_comparable"] = int(counts.get("NA", 0))
                record[f"{metric}_win_rate"] = _safe_ratio(int(counts.get("W", 0)), comparable)
                record[f"{metric}_wtl"] = (
                    f"{int(counts.get('W', 0))}/"
                    f"{int(counts.get('T', 0))}/"
                    f"{int(counts.get('L', 0))}"
                )

        for side in ("iod", "squeeze"):
            required = {f"tp_{side}", f"fp_{side}", f"fn_{side}"}
            if required <= set(group.columns):
                tp = float(group[f"tp_{side}"].sum())
                fp = float(group[f"fp_{side}"].sum())
                fn = float(group[f"fn_{side}"].sum())
                record[f"micro_precision_{side}"] = _safe_ratio(tp, tp + fp)
                record[f"micro_recall_{side}"] = _safe_ratio(tp, tp + fn)
                record[f"micro_f1_{side}"] = _safe_ratio(2 * tp, 2 * tp + fp + fn)
                record[f"micro_symmetric_difference_error_{side}"] = _safe_ratio(fp + fn, tp + fn)
            for metric in ("precision", "recall", "symmetric_difference_error", "alpha_spill"):
                column = f"{metric}_{side}"
                if column in group:
                    values = _finite_values(group[column])
                    record[f"macro_{metric}_{side}"] = (
                        float(np.mean(values)) if len(values) else math.nan
                    )
        record["delta_definition"] = "IOD safe outer minus Squeeze-u"
        record["runtime_accounting_note"] = IOD_SHARED_RUNTIME_NOTE
        records.append(record)
    return pd.DataFrame.from_records(records)


def iod_vs_squeeze_views(
    detailed: pd.DataFrame,
    *,
    pair_keys: Sequence[str] | None = None,
    summary_group_columns: Sequence[str] | None = None,
    metric_directions: Mapping[str, str] = PAIR_METRIC_DIRECTIONS,
    tie_tolerance: float = 1e-9,
    strict: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create exact case-paired IOD-safe-outer versus Squeeze-u views."""

    frame = canonicalize_detailed_results(detailed)
    iod = frame[frame["method_family"].eq("IOD") & frame["output_role"].isin(("safe_outer", "primary"))].copy()
    squeeze = frame[frame["method_family"].eq("Squeeze-u")].copy()
    pair_keys = list(pair_keys) if pair_keys is not None else _infer_pair_keys(frame)
    missing = [key for key in pair_keys if key not in frame.columns]
    if missing:
        raise ValueError(f"pair keys missing from detailed results: {missing}")
    for label, subset in (("IOD", iod), ("Squeeze-u", squeeze)):
        duplicate = subset.duplicated(pair_keys, keep=False)
        if duplicate.any():
            example = subset.loc[duplicate, pair_keys].head(3).to_dict("records")
            raise ValueError(f"{label} has duplicate rows for inferred pair keys: {example}")
    iod_key_set = set(map(tuple, iod[pair_keys].astype(object).where(iod[pair_keys].notna(), "<NA>").to_numpy()))
    squeeze_key_set = set(map(tuple, squeeze[pair_keys].astype(object).where(squeeze[pair_keys].notna(), "<NA>").to_numpy()))
    if strict and iod_key_set != squeeze_key_set:
        raise ValueError(
            "strict pairing failed: "
            f"{len(iod_key_set - squeeze_key_set)} IOD-only and "
            f"{len(squeeze_key_set - iod_key_set)} Squeeze-u-only cases"
        )

    paired = iod.merge(
        squeeze,
        on=pair_keys,
        how="inner",
        suffixes=("_iod", "_squeeze"),
        validate="one_to_one",
    )
    for metric, direction in metric_directions.items():
        iod_column, squeeze_column = f"{metric}_iod", f"{metric}_squeeze"
        if iod_column not in paired or squeeze_column not in paired:
            continue
        paired[f"delta_{metric}"] = paired[iod_column] - paired[squeeze_column]
        denominator = paired[squeeze_column].abs().replace(0, np.nan)
        paired[f"relative_delta_{metric}"] = paired[f"delta_{metric}"] / denominator
        paired[f"outcome_{metric}"] = [
            _outcome(float(a), float(b), direction, tie_tolerance)
            for a, b in zip(paired[iod_column], paired[squeeze_column])
        ]
    paired["delta_definition"] = "IOD safe outer minus Squeeze-u"
    paired["runtime_accounting_note"] = IOD_SHARED_RUNTIME_NOTE
    paired = _sort_view(paired)

    if summary_group_columns is None:
        summary_group_columns = _present(paired, DEFAULT_SUMMARY_CONTEXT)
    summary = _paired_summary(
        paired,
        list(summary_group_columns),
        metric_directions=metric_directions,
    )
    return paired, _sort_view(summary)


def all_original_algorithm_views(
    detailed: pd.DataFrame, *, include_iod_core: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return the six original method families, with IOD outer as primary."""

    frame = canonicalize_detailed_results(detailed)
    view = frame[frame["method_family"].isin(METHOD_FAMILY_ORDER)].copy()
    if not include_iod_core:
        view = view[~(view["method_family"].eq("IOD") & view["output_role"].eq("robust_core"))]
    # Unqualified legacy IOD rows are retained; modern runs use safe_outer.
    view.loc[view["method_family"].eq("IOD") & ~view["output_role"].eq("robust_core"), "method_display"] = "IOD"
    view = _sort_view(view)
    summary = summarize_results(view)
    return view, _sort_view(summary)


def matched_budget_curves(detailed: pd.DataFrame) -> pd.DataFrame:
    """Aggregate each method at each matched total-question budget."""

    frame = canonicalize_detailed_results(detailed)
    protocol_match = (
        frame["protocol"].astype(str).str.contains("matched", case=False, na=False)
        if "protocol" in frame
        else pd.Series(False, index=frame.index)
    )
    budget_match = frame["total_budget_cap"].notna() if "total_budget_cap" in frame else pd.Series(False, index=frame.index)
    matched = frame[protocol_match | budget_match]
    matched, _ = all_original_algorithm_views(matched)
    if matched.empty:
        return pd.DataFrame()
    groups = _present(
        matched,
        (
            "protocol",
            "dataset",
            "n",
            "d",
            "epsilon",
            "total_budget_cap",
            "utility_mode",
            "method_family",
            "method_display",
            "output_role",
        ),
    )
    curves = _group_records(matched, groups)
    if "total_budget_cap" in curves:
        curves["budget"] = curves["total_budget_cap"]
    curves["curve_x"] = "total question budget"
    curves["runtime_accounting_note"] = np.where(
        curves.get("method_family", pd.Series(index=curves.index, dtype=str)).eq("IOD"),
        IOD_SHARED_RUNTIME_NOTE,
        "",
    )
    return _sort_view(curves)


def power_band_ablation_summary(detailed: pd.DataFrame) -> pd.DataFrame:
    """Summarize outcomes by configured and realized hidden-power bands."""

    frame = canonicalize_detailed_results(detailed)
    if "protocol" in frame:
        explicit = frame["protocol"].astype(str).str.contains("power|band|ablation", case=False, regex=True, na=False)
        if explicit.any():
            frame = frame[explicit]
        elif frame["configured_power_gap"].dropna().nunique() < 2:
            # A single configured gap is an effectiveness experiment, not an
            # ablation.  Returning an empty view prevents an unsupported label.
            return pd.DataFrame(
                columns=[
                    "protocol",
                    "dataset",
                    "n",
                    "d",
                    "epsilon",
                    "utility_mode",
                    "configured_power_gap",
                    "method_family",
                    "method_display",
                    "output_role",
                    "runs",
                    "ablation_factor",
                ]
            )
    elif frame["configured_power_gap"].dropna().nunique() < 2:
        return pd.DataFrame(
            columns=[
                "dataset",
                "n",
                "d",
                "epsilon",
                "utility_mode",
                "configured_power_gap",
                "method_family",
                "method_display",
                "output_role",
                "runs",
                "ablation_factor",
            ]
        )
    frame, _ = all_original_algorithm_views(frame)
    if frame.empty:
        return pd.DataFrame()
    groups = _present(
        frame,
        (
            "protocol",
            "dataset",
            "n",
            "d",
            "epsilon",
            "utility_mode",
            "exponent_range_policy",
            "power_regime",
            "configured_power_gap",
            "method_family",
            "method_display",
            "output_role",
        ),
    )
    extra_metrics = tuple(
        dict.fromkeys(
            (
                *SUMMARY_METRICS,
                "selected_power_band_min",
                "selected_power_band_max",
                "selected_power_band_width",
                "selected_power_band_cardinality",
                "true_power_min",
                "true_power_max",
                "true_power_span",
            )
        )
    )
    result = _group_records(frame, groups, metrics=extra_metrics)
    result["ablation_factor"] = "maximum within-utility power gap"
    return _sort_view(result)


def build_paper_views(detailed: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Build comparison-only V3 paper tables without writing files."""

    canonical = canonicalize_detailed_results(detailed)
    pair_detail, pair_summary = iod_vs_squeeze_views(canonical)
    all_detail, all_summary = all_original_algorithm_views(canonical)
    return {
        "iod_vs_squeeze_paired_detailed": pair_detail,
        "iod_vs_squeeze_paired_summary": pair_summary,
        "all_original_algorithms_detailed": all_detail,
        "all_original_algorithms_summary": all_summary,
        "matched_budget_curves": matched_budget_curves(canonical),
        "power_band_ablation_summary": power_band_ablation_summary(canonical),
    }


def _atomic_csv(path: Path, frame: pd.DataFrame) -> None:
    temporary = path.with_name(path.name + ".partial")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def export_paper_views(
    detailed: pd.DataFrame,
    output_directory: str | Path,
) -> dict[str, Path]:
    """Write all paper views and reporting notes atomically as CSV/JSON."""

    destination = Path(output_directory)
    destination.mkdir(parents=True, exist_ok=True)
    views = build_paper_views(detailed)
    paths: dict[str, Path] = {}
    for name, frame in views.items():
        path = destination / f"{name}.csv"
        _atomic_csv(path, frame)
        paths[name] = path
    notes = {
        "reporting_schema": REPORTING_SCHEMA,
        "formal_alpha_spill": (
            "max(0, max_{p in S\\I} f(p*)-(1+epsilon)f(p)); formal_alpha is "
            "reported only when FN=0"
        ),
        "symmetric_difference_error": "(FP+FN)/|I|; this is not formal alpha spill",
        "micro_metrics": "computed after summing TP, FP, and FN across runs",
        "macro_metrics": "unweighted mean of defined per-run metrics",
        "distribution": "sample SD (0 for a singleton); IQR is Q3-Q1",
        "core_nonempty_coverage": "fraction of robust-core runs with output_size > 0",
        "iod_runtime": IOD_SHARED_RUNTIME_NOTE,
        "paired_delta": DELTA_NOTE,
        "primary_iod_comparison_output": "safe outer",
    }
    notes_path = destination / "reporting_notes.json"
    temporary = notes_path.with_name(notes_path.name + ".partial")
    temporary.write_text(json.dumps(notes, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, notes_path)
    paths["reporting_notes"] = notes_path
    return paths


__all__ = [
    "AlphaSpillResult",
    "DELTA_NOTE",
    "IOD_SHARED_RUNTIME_NOTE",
    "METHOD_FAMILY_ORDER",
    "PAIR_METRIC_DIRECTIONS",
    "REPORTING_SCHEMA",
    "all_original_algorithm_views",
    "build_paper_views",
    "canonicalize_detailed_results",
    "export_paper_views",
    "formal_alpha_spill",
    "iod_vs_squeeze_views",
    "matched_budget_curves",
    "power_band_ablation_summary",
    "summarize_results",
    "target_membership_from_scores",
]

if __name__ == "__main__":
    main()


IOD nonlinear CPU v3.1 | build=v3.1-20260723-concave-safe-r1 | profile=paper | mode=paper | powers=(1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0) | exponent_range=elicited_band | true_power_max_gap=2 | workers=4 | quick=False | hash=4c4cf2566604 | iod_preprune=elicited-band-superhomogeneous-front-v1 | concave_case_revision=concave-power-safe-front-v1
completed paper_synthetic_r0_d5 (resumed=False)
completed paper_synthetic_r0_d10 (resumed=False)
completed paper_synthetic_r0_d15 (resumed=False)
completed paper_real_r0 (resumed=False)
completed budget_r0_Q12 (resumed=False)
completed budget_r0_Q15 (resumed=False)
completed budget_r0_Q20 (resumed=False)
completed budget_r0_Q25 (resumed=False)
completed matched_dimension_r0_d5_Q17 (resumed=False)
completed matched_dimension_r0_d10_Q22 (resumed=False)
completed matched_dimension_r0_d15_Q27 (resumed=False)
completed matched_real_r0_Q16 (resumed=False)
completed power_ablation_r0_linear_control (resumed=False)
completed power_ablation_r0